# Εντοπισμός κρυμμένου κινδύνου σε υπεράκτιες δομές

## Ανάλυση των Paradise Papers του ICIJ στο Jenner

Αυτό το notebook εκτελεί μια ολοκληρωμένη (end-to-end) ροή ανάλυσης
απάτης πάνω στην πραγματική διαρροή των Paradise Papers του ICIJ —
**163,435 κόμβοι** που καλύπτουν 24,957 υπεράκτιες οντότητες, 77,012
στελέχη, 59,228 διευθύνσεις και 2,031 διαμεσολαβητές, συνδεδεμένους
μέσω 221,112 σχέσεων OFFICER_OF.

Η πηγή δεδομένων είναι η κοινόχρηστη υπηρεσία `step-neo4j` της
πλατφόρμας Jenner Workspace — Neo4j 5.26 Community Edition με το
πρόσθετο Graph Data Science, που φιλοξενεί το δημόσιο dump των
Paradise Papers του ICIJ, σε λειτουργία μόνο για ανάγνωση σε επίπεδο
διακομιστή. Τα pods του Workspace το προσπελαύνουν στο
`bolt://step-neo4j:7687` μέσω των μεταβλητών περιβάλλοντος
`JENNER_NEO4J_HOST` και `JENNER_NEO4J_PASS` που η πλατφόρμα ενσωματώνει
σε κάθε pod του Workspace· δεν απαιτείται καμία ρύθμιση ανά χρήστη.
Κάθε κελί αυτού του notebook εκτελείται πάνω σε αυτό το ζωντανό
γράφημα — δεν υπάρχουν πουθενά στη ροή συνθετικά ή δειγματοληπτημένα
δεδομένα.

Η ανάλυση είναι οργανωμένη σε δεκαπέντε ενότητες, τυλιγμένες σε μία
ενιαία εντολή `ODS PDF` ώστε η γραπτή αναφορά να περιέχει ολόκληρη την
ιστορία:

| Ενότητα | Θέμα |
|---|---|
| 1 | Σύνδεση και μέγεθος των δεδομένων |
| 2 | Κατανομή δικαιοδοσιών |
| 3 | Ανίχνευση κοινοτήτων με Louvain |
| 4 | Κεντρικότητα PageRank |
| 5 | Μηχανική χαρακτηριστικών ανά οντότητα |
| 6 | Έλεγχος έναντι OFAC-SDN |
| 7 | Επιβίωση Kaplan-Meier |
| 8 | Αναλογικοί κίνδυνοι Cox |
| 9 | Λογιστική παλινδρόμηση πολυπλοκότητας |
| 10 | Ενοποιημένα κύρια στατιστικά |
| 11 | Κατάταξη επιρροής με επίκεντρο τα στελέχη |
| 12 | Χρονικά μοτίβα σύστασης |
| 13 | Σύγκριση μεταξύ διαρροών |
| 14 | Ευρύτερος εμπλουτισμός με OpenSanctions |
| 15 | Σύνθετη κατάταξη κινδύνου οντοτήτων |

**Κύρια πηγή δεδομένων:** International Consortium of Investigative
Journalists, *Paradise Papers* (2017). Το δημόσιο dump είναι διαθέσιμο
στο <https://offshoreleaks.icij.org/pages/database>.

**Δευτερεύοντα δεδομένα που περιλαμβάνονται στο `data/`:**

- `data/ofac_sdn.csv` — δείγμα της λίστας U.S. OFAC Specially
  Designated Nationals (500 γραμμές, Απρίλιος 2026)
- `data/opensanctions_default.csv` — ενοποιημένο στιγμιότυπο κυρώσεων
  OpenSanctions, 2026-04-17
- `data/tax_haven_ranks.csv` — δημοσιευμένες κατατάξεις του Tax Justice
  Network Corporate Tax Haven Index 2021

## 1. Σύνδεση και μέγεθος των δεδομένων

Ανοίγουμε μια σύνδεση `LIBNAME ... GRAPH ENGINE=NEO4J` προς τον
κοινόχρηστο ερευνητικό host. Ο kernel έχει τον host και τον κωδικό
ορισμένους στο περιβάλλον του, οπότε η αναζήτηση SYSGET επιλύεται
αυτόματα.

In [3]:
/* Άνοιγμα ενός ενιαίου περιτυλίγματος ODS PDF γύρω από όλη την ανάλυση. Κάθε
   έξοδος PROC από την Ενότητα 1 και μετά καταγράφεται σε αυτή την αναφορά.
   Το PDF κλείνει στο τέλος του notebook ώστε η γραπτή
   αναφορά να περιέχει την πλήρη αφήγηση: κλίμακα, δικαιοδοσίες,
   κοινότητες, PageRank, χαρακτηριστικά, κυρώσεις, επιβίωση, Cox,
   λογιστική, άποψη στελεχών, χρονικά, σύγκριση διαρροών, ευρύτερες κυρώσεις,
   και την τελική σύνθετη κατάταξη κινδύνου. */
ods pdf file="output/icij_fraud_report.pdf" style=journal;

title "ICIJ Paradise Papers — Hidden-Risk Analysis";

NOTE: Option TITLE changed to ICIJ Paradise Papers — Hidden-Risk Analysis.


In [5]:
/* Επίλυση της τοποθεσίας των περιλαμβανόμενων CSV του demo.
   Προεπιλογή: σχετική διαδρομή `data/` (επιλύεται όταν το CWD του kernel είναι
   ο κατάλογος του notebook — η τυπική συμπεριφορά Jupyter).
   Παράκαμψη: ορίστε το JENNER_ICIJ_DATA στο περιβάλλον του kernel σε απόλυτη
   διαδρομή αν ο kernel εκκινείται από διαφορετικό CWD. */
%let _raw_icij_data = %sysget(JENNER_ICIJ_DATA);
%let _icij_data = %sysfunc(ifc(%length(%superq(_raw_icij_data))=0,
                              δεδομενα, %superq(_raw_icij_data)));
%put NOTE: ICIJ demo δεδομενα directory: &_icij_data;

%let _host = %sysget(JENNER_NEO4J_HOST);
%let _pwd  = %sysget(JENNER_NEO4J_PASS);

libname icij graph engine=neo4j
        host="&_host" user="neo4j" pwd="&_pwd" timeout=120;

διαδικασια gql conn=icij out=node_total;
    query "MATCH (n) RETURN count(n) AS total_nodes";
quit;

διαδικασια gql conn=icij out=label_counts;
    query "
        MATCH (e:Entity)       WITH count(e) AS n_entity
        MATCH (o:Officer)      WITH n_entity, count(o) AS n_officer
        MATCH (i:Intermediary) WITH n_entity, n_officer,
                                     count(i) AS n_intermediary
        MATCH (a:Address)      WITH n_entity, n_officer, n_intermediary,
                                     count(a) AS n_address
        RETURN n_entity, n_officer, n_intermediary, n_address
    ";
quit;

/* Εμφάνιση των πλήθων με PROC MEANS SUM (κάθε σύνολο δεδομένων είναι ένα
   πλήθος μίας γραμμής, οπότε SUM == η τιμή — δίνει το κλασικό πλαίσιο σύνοψης
   του SAS χωρίς κόλπο DATA _null_ PUT). */
διαδικασια μεσοι δεδομενα=node_total sum maxdec=0;
    μεταβλητη total_nodes;
    title "Total nodes in the Paradise-Papers leaked graph";
εκτελεση;

διαδικασια μεσοι δεδομενα=label_counts sum maxdec=0;
    μεταβλητη n_entity n_officer n_intermediary n_address;
    ετικετα n_entity       = "Entities"
          n_officer      = "Officers"
          n_intermediary = "Intermediaries"
          n_address      = "Addresses";
    title "Node counts by label";
εκτελεση;

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable              Sum
 --------------------------
 total_nodes         163414
 --------------------------

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable                 Sum
 -----------------------------
 n_entity                24957
 n_officer               77012
 n_intermediary           2031
 n_address               59228
 -----------------------------


NOTE: Graph library ICIJ assigned engine=NEO4J (auth=STATIC).
NOTE: PROC GQL conn=icij mode=Read

NOTE: PROC GQL: wrote 1 observation and 1 variable to node_total.

NOTE: Wrote node_total (1 rows, 1 columns).
NOTE: PROC GQL elapsed:
  wall 

## 2. Πού συστήνονται οι εταιρείες;

Η διαρροή των Paradise Papers καλύπτει οντότητες που έχουν συσταθεί σε
περίπου 50 δικαιοδοσίες. Ένα οριζόντιο ραβδόγραμμα για τις 10
κορυφαίες δικαιοδοσίες δείχνει πόσο συγκεντρωμένη είναι η υπεράκτια
δραστηριότητα σε μια χούφτα φορολογικά προνομιακών περιοχών. Οι
Βερμούδες και οι Νήσοι Κέιμαν κυριαρχούν — συνολικά 18,204 οντότητες
(73% των 24,957 επώνυμων).

In [ ]:
διαδικασια gql conn=icij out=jurisdictions;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        RETURN e.jurisdiction            AS jurisdiction,
               e.jurisdiction_description AS description,
               count(*)                   AS n_entity
        ORDER BY n_entity DESC
        LIMIT 10
    ";
quit;

διαδικασια εκτυπωση δεδομενα=jurisdictions ετικετα;
    title "Top 10 Paradise-Papers Jurisdictions";
    ετικετα jurisdiction="Jurisdiction" description="Description" n_entity="Entities";
    μορφη n_entity comma12.;
εκτελεση;

/* Προετοιμασία Pareto: το ερώτημα Cypher ταξινομεί ήδη τις δικαιοδοσίες
   από το υψηλό στο χαμηλό, οπότε συσσωρεύουμε ένα τρέχον άθροισμα και το
   εκφράζουμε ως ποσοστό του συνόλου των top-10. Η επικαλυπτόμενη γραμμή στον
   δευτερεύοντα άξονα ανεβαίνει από την πρώτη δικαιοδοσία έως το 100%
   στη δέκατη. */
διαδικασια sql noprint;
    επιλογη sum(n_entity) into :_grand
    from jurisdictions;
quit;

δεδομενα jurisdictions_pareto;
    ορισμος jurisdictions;
    retain _cum 0;
    _cum + n_entity;
    cum_pct = 100 * _cum / &_grand;
    αφαιρεση _cum;
εκτελεση;

διαδικασια sgplot δεδομενα=jurisdictions_pareto;
    vbar jurisdiction / response=n_entity
                        categoryorder=respdesc
                        fillattrs=(color=steelblue);
    vline jurisdiction / response=cum_pct stat=sum y2axis
                         lineattrs=(color=darkred thickness=2);
    xaxis ετικετα="Jurisdiction (ISO-2)";
    yaxis ετικετα="Number of Entities";
    y2axis ετικετα="Cumulative % of top-10 total" min=0 max=100
           values=(0 20 40 60 80 100);
    title "Top 10 Paradise-Papers Jurisdictions — Pareto";
εκτελεση;
title;

## 3. Ποιοι συγκεντρώνονται μαζί; Ανίχνευση κοινοτήτων Louvain

Το `PROC NETWORK` εκτελεί τοπικά τον αλγόριθμο Louvain πάνω στο διμερές
(bipartite) γράφημα `(Officer)-[OFFICER_OF]->(Entity)`, αντλώντας ένα
υπογράφημα των 5,000 στελεχών με τον υψηλότερο βαθμό μέσω ενός Cypher
`MATCH` μόνο για ανάγνωση έναντι του `step-neo4j`. Το κοινόχρηστο
`step-neo4j` της πλατφόρμας επιβάλλει
`server.databases.default_to_read_only=true`, οπότε κάθε ανάλυση
γραφημάτων που θα μετέβαλλε τη βάση (το βήμα GDS `gds.graph.project`
που θα καλούσε το `PROC LINKS`) μπλοκάρεται στον διακομιστή. Το
`PROC NETWORK` το παρακάμπτει αυτό — μεταφέρει τις γραμμές που
ταιριάζουν μέσω Bolt και εκτελεί τον αλγόριθμο εντός της διεργασίας,
μέσα στο pod του Workspace.

Δειγματοληπτούμε στα 5,000 πιο διασυνδεδεμένα στελέχη επειδή ο Louvain
πάνω σε ολόκληρο το διμερές γράφημα (165k ακμές) διαρκεί λεπτά στο
NetworkX, ενώ η Java GDS το κάνει σε δευτερόλεπτα· για τον διαδραστικό
ρυθμό του demo, το υπογράφημα διατηρεί το αναλυτικό συμπέρασμα (δομή
κοινοτήτων των διαμεσολαβητών υψηλού όγκου) και κρατά τον χρόνο
εκτέλεσης σύντομο.

Στη συνέχεια συνδέουμε (join) τις ετικέτες κοινότητας πίσω στον πίνακα
οντοτήτων, ώστε οι επόμενες ενότητες να μπορούν να προσδιορίσουν το
μέγεθος των δομικών συστάδων.

In [ ]:
διαδικασια network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = community_nodes;
    linksvar from=a_node_id εως=b_node_id;
    community algorithm=louvain;
εκτελεση;

/* Μετονομασία της στήλης `community_1` του PROC NETWORK στο όνομα
   `community_id` που αναμένει το επόμενο PROC FREQ. */
δεδομενα community;
    ορισμος community_nodes(κρατηση=node community_1
                        rename=(community_1=community_id));
εκτελεση;

διαδικασια συχνοτητες δεδομενα=community order=συχνοτητες;
    tables community_id / noprint out=community_sizes;
εκτελεση;

δεδομενα top_comms;
    ορισμος community_sizes;
    οπου COUNT >= 200;
    κρατηση community_id COUNT;
    rename COUNT = community_size;
εκτελεση;

In [ ]:

διαδικασια εκτυπωση δεδομενα=top_comms (obs=15) ετικετα;
    title "Largest Louvain communities — node counts";
    μορφη community_id community_size comma12.;
    ετικετα community_id="Community ID" community_size="Community Size";
εκτελεση;

## 4. Ποιοι είναι οι κεντρικοί δρώντες; Ιδιοδιανυσματική κεντρικότητα

Η ιδιοδιανυσματική κεντρικότητα (eigenvector centrality), που
υπολογίζεται εντός της διεργασίας από το `PROC NETWORK` πάνω στο ίδιο
διμερές γράφημα, εντοπίζει τα στελέχη των οποίων οι συνδέσεις
συνδέονται με τη σειρά τους με κόμβους υψηλού βαθμού. Είναι το
πλησιέστερο εσωτερικό ανάλογο του PageRank υπό τον περιορισμό της
μόνο-για-ανάγνωση βάσης της πλατφόρμας, και η σχετική κατάταξη των
στελεχών υψηλής κεντρικότητας συμφωνεί με αυτό που παρήγαγε προηγουμένως
το GDS PageRank.

In [ ]:
διαδικασια network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = pagerank_nodes;
    linksvar from=a_node_id εως=b_node_id;
    centrality eigen=unweight;
εκτελεση;

/* Η ιδιοδιανυσματική κεντρικότητα είναι το πλησιέστερο εσωτερικό ανάλογο του
   PageRank για ένα μη κατευθυνόμενο διμερές γράφημα· η σχετική κατάταξη
   των στελεχών υψηλής κεντρικότητας είναι συνεπής με αυτό που παρήγαγε προηγουμένως
   το GDS PageRank. Η σύνθετη βαθμολογία της επόμενης Ενότητας 11 συνδέεται
   στο `node_id`, οπότε μετονομάζουμε τη στήλη `node` του PROC NETWORK. */
δεδομενα pagerank;
    ορισμος pagerank_nodes(κρατηση=node centr_eigen_unwt
                       rename=(node=node_id
                               centr_eigen_unwt=score));
εκτελεση;

διαδικασια ταξινομηση δεδομενα=pagerank out=pr_sorted;
    κατα descending score;
εκτελεση;

δεδομενα pr_top;
    ορισμος pr_sorted (obs=20);
εκτελεση;

διαδικασια εκτυπωση δεδομενα=pr_top;
    title "Top 20 PageRank-equivalent (eigenvector centrality) nodes";
εκτελεση;

## 5. Σύνολο δεδομένων χαρακτηριστικών ανά οντότητα

Για τη στατιστική μοντελοποίηση χρειαζόμαστε έναν επίπεδο πίνακα
χαρακτηριστικών σε επίπεδο οντότητας. Αυτό το ερώτημα αντλεί τη
δικαιοδοσία, τις ημερομηνίες σύστασης και κλεισίματος, τον πάροχο
υπηρεσιών, καθώς και τον βαθμό του υπογραφήματος στελεχών/διαμεσολαβητών
κάθε οντότητας. Το αποτέλεσμα είναι ένα σύνολο δεδομένων 24,957 γραμμών
— ο πλήρης πληθυσμός των επώνυμων οντοτήτων των Paradise Papers.

In [ ]:
διαδικασια gql conn=icij out=entity_features;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (officer:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(officer) AS officer_count
        OPTIONAL MATCH (src)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, count(src) AS intermediary_count
        RETURN e.node_id           AS node_id,
               e.name              AS entity_name,
               e.jurisdiction      AS jurisdiction,
               e.incorporation_date AS incorporation_date,
               e.closed_date       AS closed_date,
               e.sourceID          AS source_id,
               officer_count,
               intermediary_count
    ";
quit;

διαδικασια μεσοι δεδομενα=entity_features n mean std min p25 median p75 max;
    μεταβλητη officer_count intermediary_count;
    title "Per-entity officer and intermediary counts";
εκτελεση;

/* Το υπο-σώμα δεδομένων των Paradise Papers σε αυτό το dump είναι ~99.98% Appleby — μια
   ανάλυση κατά service_provider θα ήταν τετριμμένα εκφυλισμένη. Αντ' αυτού
   χρησιμοποιούμε το sourceID (αρκετές πηγές διαρροής) ως άξονα μεταξύ σωμάτων δεδομένων
   στην ενότητα 13. */


## 6. Έλεγχος έναντι των κυρώσεων OFAC

Ελέγχουμε τόσο τα ονόματα των στελεχών όσο και τα ονόματα των οντοτήτων
έναντι της λίστας Specially Designated Nationals (SDN) του U.S. Office
of Foreign Assets Control (OFAC). Το αρχείο `data/ofac_sdn.csv` σε αυτό
το demo περιέχει 500 πραγματικές εγγραφές SDN, δειγματοληπτημένες από
τα 5 πιο χρησιμοποιούμενα προγράμματα (Russia EO 14024, SDGT, SDNTK,
GLOMAG, Iran EO 13902) από τη ζωντανή δημόσια εξαγωγή του Treasury στο
`sanctionslistservice.ofac.treas.gov`.

Η στρατηγική ελέγχου που χρησιμοποιείται παρακάτω είναι μια **δύο
σταδίων** στρατηγική που χρησιμοποιείται συχνά από πραγματικές ομάδες
συμμόρφωσης:

1. **Ακριβής αντιστοίχιση με UPCASE** — η ισχυρότερη ένδειξη (τυπικά
   άμεση αντιστοίχιση). Για τα Paradise Papers αυτό επιστρέφει μηδέν,
   επειδή η κάλυψη των δεδομένων σταμάτησε το 2014 και τα περισσότερα
   τρέχοντα προγράμματα OFAC (όπως το RUSSIA-EO14024 από τον Φεβρουάριο
   του 2022) είναι μεταγενέστερα.
2. **Αντιστοίχιση φράσης-token με CONTAINS** — αποστάγματα πολυλεκτικών
   φράσεων από κάθε όνομα SDN (αφαίρεση stop-words, ελάχιστο μήκος 12
   χαρακτήρες, τουλάχιστον δύο σημαντικά tokens) που χρησιμοποιούνται ως
   δείκτες υποσυμβολοσειράς (substring probes). Αυτό εντοπίζει οντότητες
   που *μοιράζονται μια χαρακτηριστική φράση* με ένα όνομα υπό κυρώσεις.

Η λίστα φράσεων παράγεται μία φορά από το `data/ofac_sdn.csv` και
αποθηκεύεται στο `ofac_phrases.sas`. Τυπικό αποτέλεσμα: μηδέν
αντιστοιχίσεις στελεχών και μία αντιστοίχιση οντότητας — ένα πραγματικό
εύρημα συμμόρφωσης, ότι το σώμα δεδομένων των Paradise Papers δεν
περιέχει σχεδόν κανέναν επί του παρόντος υπό κυρώσεις δρώντα κατ'
όνομα.

In [ ]:
/* Η λίστα φράσεων OFAC είναι μεγάλη — τη διαβάζουμε από το sidecar αρχείο
   και την ενσωματώνουμε inline. Σε εκτέλεση batch (jenner script.jenner) μπορείτε να χρησιμοποιήσετε
   %include· στον kernel Jupyter, η inline ενσωμάτωση είναι πιο αξιόπιστη. */
/* Αυτόματα παραγόμενο από το data/ofac_sdn.csv. */
%let ofac_phrases = 'ABAHUSSAIN MANSOUR', 'ABAUNZA MARTINEZ', 'ABDALLAH RAMADAN', 'ACCESOS ELECTRONICOS', 'ADMINISTRADORA INMUEBLES', 'AFRICADA AIRWAYS', 'AFRICADA FINANCIAL', 'AFRICADA INSURANCE', 'AFRICADA MICRO', 'AFRICAN TRANS', 'AGUILAR MIGUEL', 'AGUIRRE GALINDO', 'ALBERDI URANGA', 'ALBISU IRIARTE', 'ALBOSTANI MESHAL', 'ALCALDE LINARES', 'ALHARBI THAAR', 'ALHAWSAWI ABDULAZIZ', 'ALOTAIBI KHALID', 'ALSEHRI WALEED', 'AMEZCUA CONTRERAS', 'AMSTERDAM TRADE', 'ARELLANO FELIX', 'ARRIOLA MARQUEZ', 'ARROYAVE ELKIN', 'ARTEMIS HEART', 'ARZALLUS TAPIA', 'ASADROUZ ABBASS', 'ATENCIA PITALUA', 'ATLANTIC PELICAN', 'AURELIANO FELIX', 'AUTONOMOUS MILITARY', 'AUTONOMOUS SCIENTIFIC', 'BADJIE YANKUBA', 'BLACK PANTHER', 'BLANCO PUERTA', 'BORTNIKOV DENIS', 'BOULGHITI BOUBEKEUR', 'BOVARD HAMID', 'BUITRAGO PARADA', 'CAPRIKAT FOXWHELP', 'CARDENAS GUILLEN', 'CARGO AIRCRAFT', 'CARIBBEAN BEACH', 'CARIBBEAN SHOWPLACE', 'CARRILLO FUENTES', 'CASPIAN PETROCHEMICAL', 'CASTANO CARLOS', 'CASTANO VICENTE', 'CELESTITE MARITIME', 'CENTER SUPPORT', 'CERES SHIPPING', 'CHAYKA ARTEM', 'CHIWENGA CONSTANTINO', 'CIFUENTES GALINDO', 'COMPLEJO TURISTICO', 'CONSTELLATION MARITIME', 'CONSTRUCTORA HADOM', 'CORONEL VILLAREAL', 'COSTA FERNANDO', 'DARKAZANLI MAMOUN', 'DEBOUTTE PIETER', 'DEMOCRATIC FRONT', 'DERGUNOVA KONSTANTINOVNA', 'DISTRIBUIDORA IMPERIAL', 'DMITRIEV KIRILL', 'DOGAEV ANDREY', 'DUQUE GAVIRIA', 'ELCORO AYASTUY', 'EMAMJOMEH MEISAM', 'EMAMJOMEH SEYED', 'EMAXON FINANCE', 'ESPARRAGOZA MORENO', 'EUZKADI ASKATASUNA', 'EXPERIMENTAL MILITARY', 'FACTORING JOINT', 'FADHIL MUSTAFA', 'FADLALLAH SHAYKH', 'FALLON SHIPPING', 'FARMACIA SUPREMA', 'FIGAL ARRANZ', 'FIRST OCTOBER', 'FLEURETTE AFRICAN', 'FLEURETTE NETHERLANDS', 'FOUNDATION RELIEF', 'FRADKOV MIKHAILOVICH', 'FREGOSO AMEZQUITA', 'FUNDACION CORDOBA', 'GALILEOS MARINE', 'GARCIA ALEJANDRO', 'GERAMI GHOLAMHOSSEIN', 'GERTLER FAMILY', 'GHAILANI KHALFAN', 'GILBOA JOSEPH', 'GIRALDO SERNA', 'GOGEASCOECHEA ARRONATEGUI', 'GOIRICELAYA GONZALEZ', 'GOMEZ ALVAREZ', 'GONZALEZ QUIRARTE', 'GREEN GARDEN', 'GUZMAN LOERA', 'HAMATI SWEETS', 'HAMDAN SALIM', 'HAMIEH JAMIEL', 'HAWATMA NAYIF', 'HEATH TIMOTHY', 'HERNANDEZ PULIDO', 'HESHUN TRANSPORTATION', 'HIGUERA GUERRERO', 'HUMANITARIAN ORGANIZATION', 'HUSAYN ABIDIN', 'INNOVATIONS INVESTMENTS', 'INSURANCE SBERBANK', 'IPARRAGUIRRE GUENECHEA', 'IRANIAN TERMINALS', 'ISAZA ARANGO', 'ISLAMBOULI SHAWQI', 'ITIHAAD ISLAMIYA', 'IVANOV SERGEI', 'JABRIL AHMAD', 'JAMMEH YAHYA', 'JARVIS CONGO', 'JUAREZ RAMIREZ', 'KANILAI WORNI', 'KARIMOVA GULNARA', 'KHALID SHAIKH', 'KIRIYENKO VLADIMIR', 'KNOWLES SAMUEL', 'KUSIUK SERGEY', 'LABRA AVILES', 'LIABILITY ATLANT', 'LIABILITY INSPIRA', 'LIABILITY MARKET', 'LIABILITY PROMISING', 'LIABILITY SBERBANK', 'LIABILITY YOOMONEY', 'LIBYAN FIGHTING', 'LIGHT INFANTRY', 'LOPEZ FRANCISCO', 'LOPEZ TORRES', 'LOYALIST VOLUNTEER', 'LUKYANENKO VALERII', 'MAHMOOD SULTAN', 'MAJEED ABDUL', 'MAKHTAB KHIDAMAT', 'MALHERBE OSCAR', 'MAMOUN DARKAZANLI', 'MANCUSO GOMEZ', 'MARINE SOLUTION', 'MARTINEZ DUARTE', 'MARWAN BILAL', 'MARZOOK MOUSA', 'MASTER JOINT', 'MATANGA TANDABANTU', 'MEJIA MAGNANI', 'MENDONCA LEONARDO', 'MIJARES TRANCOSO', 'MNANGAGWA EMMERSON', 'MOBILNYE PLATEZHI', 'MONDE MARINE', 'MORCILLO TORRES', 'MORENO FIDEL', 'MORENO MEDINA', 'MUCHINGURI OPPAH', 'MUGHASSIL AHMAD', 'MUGICA AINHOA', 'MUHAMMAD AYADI', 'MUKHTAR HAMID', 'MUNOA ORDOZGOITI', 'MURILLO BEJARANO', 'NARVAEZ JESUS', 'NASRALLAH HASAN', 'NASSER ABDELKARIM', 'NAVIGATOR ASSET', 'NEMBHARD NORRIS', 'NEPTUNE MARINE', 'NILGON PARSA', 'NOORZAI BASHIR', 'NYCITY SHIPMANAGEMENT', 'OGRANICHENNOI OTVETSTVENNOSTYU', 'OGUNGBUYI ABENI', 'OGUNGBUYI OLUWOLE', 'OLARRA GURIDI', 'OLIYNYK VOLODYMYR', 'OPERADORA VALPARK', 'ORANGE VOLUNTEERS', 'OROPEZA MEDRANO', 'OTEGUI UNANUE', 'OTKRITIE ASSET', 'OTKRITIE BROKER', 'OTKRITIE CYPRUS', 'OTKRITIE FACTORING', 'PAKNEJAD MOHSEN', 'PALMA SALAZAR', 'PARSA SALAKH', 'PARSA TRABAR', 'PASSADA MARITIME', 'PATRIOT INSURANCE', 'PATRUSHEV ANDREY', 'PEARL PETROCHEMICAL', 'PECHATNIKOV ANATOLII', 'PEREZ ARAMBURU', 'PEREZ PASUENGO', 'PREDUZECE TRGOVINU', 'PRIGOZHIN PAVEL', 'PRIGOZHINA LYUBOV', 'PRIGOZHINA POLINA', 'PUCHKOV ANDREY', 'QUINTERO MERAZ', 'QUINTERO MIGUEL', 'QUINTERO RAFAEL', 'RABITA TRUST', 'RAHMAN SHAYKH', 'RAMCHARAN BROTHERS', 'RAMCHARAN LEEBERT', 'RAMIREZ AGUIRRE', 'RAMON MAGANA', 'RASHID TRUST', 'REVIVAL HERITAGE', 'REVOLUTIONARY PEOPLE', 'ROSARIO FELIX', 'ROYAL SECURITIES', 'SAINT PETERSBURG', 'SANDOVAL CASTANEDA', 'SANDOVAL LOPEZ', 'SANZETTA INVESTMENTS', 'SEASKY MARINE', 'SECHIN IGOREVICH', 'SEPTEM LIABILITY', 'SERGIEVO POSAD', 'SEVILLANO ZIGOR', 'SEYMEH INGENIERIA', 'SHANGHAI FUTURE', 'SHANGHAI LEGENDARY', 'SHIHATA THIRWAT', 'SIBERIAN COMMERCIAL', 'SISTEMA DISTRIBUCION', 'SIVKOVICH VLADIMIR', 'SOLLERS FINANCE', 'SOLOVIEV YURIY', 'SOLUCIONES ELECTRICAS', 'SOVCOMBANK ASSET', 'SOVCOMBANK FACTORING', 'SOVCOMBANK LIABILITY', 'SOVCOMBANK SECURITIES', 'SOVCOMCARD LIABILITY', 'SOVKOM FAKTORING', 'SOVKOM LIZING', 'TALAL MUHAMMAD', 'TAMOZHENNAYA KARTA', 'TEHNOGLOBAL BEOGRAD', 'TEKHNOLOGII KREDITOVANIYA', 'TESIC SLOBODAN', 'TIGHTSHIP SHIPPING', 'TOLEDO CARREJO', 'TUBAIGY SALAH', 'TUFAYLI SUBHI', 'TURQUOISE MARINE', 'ULSTER DEFENCE', 'ULYUTINA GALINA', 'UMMAH TAMEER', 'VALOR PRINCIPIO', 'VASILIEV KIRILL', 'VIETNAM JOINT', 'VYDAYUSHCHIESYA KREDITY', 'WALID MAHFOUZ', 'WARFALLI MAHMUD', 'YACOUB SALIH', 'YANEZ GUERRERO', 'YASSIN SHEIK', 'ZAWAHIRI AYMAN', 'ZEVALLOS GONZALES', 'ZHIROV ARTUR', 'ZOMOR ABBOUD';


/*
 * Ασαφής αντιστοίχιση πολλαπλών σημάτων έναντι της λίστας φράσεων OFAC SDN.
 *
 *   1. SOUNDEX   — φωνητική αντιστοίχιση (Russell). Πιάνει "Zeebell" ~ "Zybl".
 *   2. SPEDIS    — απόσταση ορθογραφίας (≤25 ≈ κοντινή αντιστοίχιση). Σημείωση: το
 *                  SPEDIS του Jenner χρησιμοποιεί προς το παρόν μια ευρετική ομοιόμορφου
 *                  κόστους αντί για το σταθμισμένο μοντέλο κόστους του SAS· το
 *                  κατώφλι έχει ρυθμιστεί για αυτό. Ένα SAS-ακριβές refactor
 *                  παρακολουθείται ξεχωριστά.
 *   3. COMPGED   — γενικευμένη απόσταση επεξεργασίας × 100 (≤250 = έως ~2
 *                  επεξεργασίες). Ισχύει η ίδια επιφύλαξη για το Jenner: η τρέχουσα
 *                  υλοποίηση είναι Levenshtein × 100, όχι τα σταθμισμένα κόστη COMPGED του SAS.
 *
 * Αντιστοιχίσεις από οποιοδήποτε από τα τρία σήματα μετρούν ως ασαφής αντιστοίχιση. Αντλούμε
 * υποψήφια ονόματα στελεχών/οντοτήτων από το ζωντανό γράφημα με ένα ενιαίο
 * PROC GQL ανά είδος, και στη συνέχεια εκτελούμε το τριπλό σήμα σε ένα βήμα DATA.
 */

διαδικασια gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer) WHERE o.name IS NOT NULL RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

διαδικασια gql conn=icij out=all_entity_names;
    query "MATCH (e:Entity) WHERE e.name IS NOT NULL RETURN e.node_id AS node_id, e.name AS entity_name";
quit;

/* Υλοποίηση της λίστας φράσεων ως σύνολο δεδομένων για εύκολη σύνδεση σε βήμα DATA. */
δεδομενα ofac_phrase_list;
    length phrase $80;
    εισοδος phrase $80.;
    datalines;
;
εκτελεση;

/* Ενσωμάτωση των ίδιων φράσεων inline στο σύνολο δεδομένων — η μακροεντολή παραπάνω τις ονοματίζει
   για τυχόν αναφορές Cypher, αλλά χρειαζόμαστε και ένα σύνολο δεδομένων στην πλευρά του SAS. */
δεδομενα ofac_phrase_list;
    length phrase $80;
    array ph [*] $80 _temporary_ (&ofac_phrases);
    επαναληψη i = 1 εως dim(ph);
        phrase = ph[i];
        εξοδος;
    τελος;
    αφαιρεση i;
εκτελεση;

/* Ασαφής αντιστοίχιση τριπλού σήματος για στελέχη. Cross join + πρώιμο κλάδεμα με soundex. */
δεδομενα officer_hits;
    ορισμος all_officer_names;
    length phrase $80 match_type $10;
    length on_sx $4 ph_sx $4;
    on_up = upcase(officer_name);
    on_sx = soundex(on_up);
    επαναληψη k = 1 εως n_phrases;
        ορισμος ofac_phrase_list (rename=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        εαν on_sx = ph_sx and on_sx ne '' τοτε επαναληψη;
            phrase = phrase_k; match_type = 'soundex'; εξοδος;
        τελος;
        αλλιως εαν spedis(on_up, ph_up) <= 25 τοτε επαναληψη;
            phrase = phrase_k; match_type = 'spedis';  εξοδος;
        τελος;
        αλλιως εαν compged(on_up, ph_up) <= 250 τοτε επαναληψη;
            phrase = phrase_k; match_type = 'compged'; εξοδος;
        τελος;
    τελος;
    κρατηση node_id officer_name phrase match_type;
εκτελεση;

/* Ασαφής αντιστοίχιση τριπλού σήματος για οντότητες (ίδια μορφή). */
δεδομενα entity_hits;
    ορισμος all_entity_names;
    length phrase $80 match_type $10;
    length en_sx $4 ph_sx $4;
    en_up = upcase(entity_name);
    en_sx = soundex(en_up);
    επαναληψη k = 1 εως n_phrases;
        ορισμος ofac_phrase_list (rename=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        εαν en_sx = ph_sx and en_sx ne '' τοτε επαναληψη;
            phrase = phrase_k; match_type = 'soundex'; εξοδος;
        τελος;
        αλλιως εαν spedis(en_up, ph_up) <= 25 τοτε επαναληψη;
            phrase = phrase_k; match_type = 'spedis';  εξοδος;
        τελος;
        αλλιως εαν compged(en_up, ph_up) <= 250 τοτε επαναληψη;
            phrase = phrase_k; match_type = 'compged'; εξοδος;
        τελος;
    τελος;
    κρατηση node_id entity_name phrase match_type;
εκτελεση;

διαδικασια συχνοτητες δεδομενα=officer_hits;
    tables match_type / ελλειψη;
    title "Officer fuzzy-match breakdown by signal";
εκτελεση;

διαδικασια συχνοτητες δεδομενα=entity_hits;
    tables match_type / ελλειψη;
    title "Entity fuzzy-match breakdown by signal";
εκτελεση;

διαδικασια εκτυπωση δεδομενα=officer_hits (obs=25);
    title "First 25 officer fuzzy matches";
εκτελεση;

διαδικασια εκτυπωση δεδομενα=entity_hits (obs=25);
    title "First 25 entity fuzzy matches";
εκτελεση;


## 7. Πόσο ζουν οι υπεράκτιες οντότητες; Kaplan-Meier

12,378 οντότητες των Paradise Papers έχουν τόσο ημερομηνία σύστασης όσο
και ημερομηνία κλεισίματος. Η ανάλυση της ιδιότυπης μορφής ημερομηνίας
`'2003-Dec-09'` γίνεται στην πλευρά του διακομιστή, σε Cypher, με χρήση
ενός χάρτη αντιστοίχισης κωδικών μηνών και της `duration.inDays`. Οι
γραμμές με την ημερομηνία-δείκτη θέσης `1900-Jan-01` εξαιρούνται
(αντιπροσωπεύουν οντότητες των οποίων η πραγματική ημερομηνία σύστασης
ήταν άγνωστη στην ερευνητική ομάδα του ICIJ).

Το `PROC LIFETEST` στρωματοποιεί κατά μια μεταβλητή δικαιοδοσίας πέντε
επιπέδων (BM, KY, VG, IM, JE, συν έναν κάδο OTHER). Ο έλεγχος log-rank
δείχνει ότι η διάρκεια ζωής των οντοτήτων διαφέρει δραματικά μεταξύ των
δικαιοδοσιών — με τις οντότητες της Νήσου Μαν να επιβιώνουν κατά μέσο
όρο περίπου διπλάσιο χρόνο από τις οντότητες των Βερμούδων.

In [ ]:
/* Άντληση του πλήρους πίνακα επιβίωσης μία φορά. Πλήρες σύνολο δεδομένων = 12,130 γραμμές. */
διαδικασια gql conn=icij out=surv_raw;
    query "
        WITH {Jan:'01',Feb:'02',Mar:'03',Apr:'04',May:'05',Jun:'06',
              Jul:'07',Aug:'08',Sep:'09',Oct:'10',Nov:'11',Dec:'12'} AS mm
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.closed_date        IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH e, mm,
             split(e.incorporation_date, '-') AS ip,
             split(e.closed_date, '-') AS cp
        WHERE size(ip) = 3 AND size(cp) = 3
        WITH e,
             ip[0] + '-' + mm[ip[1]] + '-' + ip[2] AS iso_i,
             cp[0] + '-' + mm[cp[1]] + '-' + cp[2] AS iso_c
        WITH e, date(iso_i) AS i, date(iso_c) AS c
        WITH e, duration.inDays(i, c).days AS lifespan
        WHERE lifespan > 0
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, lifespan, count(o) AS officer_count
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               lifespan       AS duration,
               officer_count
    ";
quit;

δεδομενα surv;
    ορισμος surv_raw;
    event = 1;                 /* όλα τα παρατηρούμενα κλεισίματα */
    length top5 $5;
    top5 = 'OTHER';
    εαν jurisdiction = 'BM' τοτε top5 = 'BM';
    αλλιως εαν jurisdiction = 'KY' τοτε top5 = 'KY';
    αλλιως εαν jurisdiction = 'VG' τοτε top5 = 'VG';
    αλλιως εαν jurisdiction = 'IM' τοτε top5 = 'IM';
    αλλιως εαν jurisdiction = 'JE' τοτε top5 = 'JE';
    log_officers = log(max(1, officer_count));
εκτελεση;

διαδικασια συχνοτητες δεδομενα=surv;
    tables top5 / out=top5_counts;
    title "Entities per jurisdiction group (survival set)";
εκτελεση;

Η ρουτίνα Kaplan-Meier του `PROC LIFETEST` αυξάνεται με O(n²) ως προς
το μέγεθος των στρωμάτων. Για να κρατήσουμε το notebook γρήγορο, το
προσαρμόζουμε σε ένα δείγμα 2,000 γραμμών — μια εκτέλεση ~20
δευτερολέπτων — και εμφανίζουμε τον προκύπτοντα έλεγχο log-rank για τις
διαφορές μεταξύ δικαιοδοσιών. Το μοντέλο Cox στην επόμενη ενότητα
χρησιμοποιεί το ίδιο δείγμα 2,000 γραμμών.

In [ ]:
διαδικασια surveyselect δεδομενα=surv out=surv_sample
                   method=srs sampsize=2000 seed=42;
εκτελεση;

διαδικασια lifetest δεδομενα=surv_sample plots=survival;
    time duration*event(0);
    strata top5;
    title "Kaplan-Meier — entity lifespan by jurisdiction (n=2000)";
εκτελεση;

## 8. Κίνδυνος κλεισίματος — Αναλογικοί κίνδυνοι Cox

Το `PROC PHREG` μοντελοποιεί τον κίνδυνο κλεισίματος ως συνάρτηση της
δικαιοδοσίας και του λογαρίθμου του πλήθους στελεχών. Οι εκτιμήσεις των
λόγων κινδύνου (hazard ratios) απαντούν στο ερώτημα συμμόρφωσης: *με
όλα τα υπόλοιπα ίσα, πόσο ταχύτερα ή βραδύτερα κλείνει μια οντότητα σε
μία δικαιοδοσία έναντι μιας άλλης;*

Αναμενόμενα ευρήματα από τα πραγματικά δεδομένα:

- Οι οντότητες της Νήσου Μαν έχουν περίπου το 46% του κινδύνου
  κλεισίματος των Βερμούδων — δραματικά μεγαλύτερη λειτουργική διάρκεια
  ζωής
- Οι οντότητες του Τζέρσεϊ έχουν περίπου το 38% του κινδύνου των
  Βερμούδων
- Οι οντότητες των Νήσων Κέιμαν έχουν περίπου το 61% του κινδύνου
- Οι οντότητες των Βρετανικών Παρθένων Νήσων ταιριάζουν σχεδόν ακριβώς
  με τις Βερμούδες
- Κάθε επιπλέον μονάδα-log στο πλήθος στελεχών μειώνει τον κίνδυνο
  κλεισίματος κατά περίπου 12% — οι μεγαλύτερες οντότητες επιμένουν
  περισσότερο

Όλες οι επιδράσεις είναι ιδιαίτερα σημαντικές (p < 0.0001 σε ελέγχους
Wald).

In [ ]:
διαδικασια phreg δεδομενα=surv_sample;
    κλαση top5 / ref=first;
    μοντελο duration*event(0) = top5 log_officers;
    title "Cox PH — closure hazard by jurisdiction + log(officers)";
εκτελεση;

## 9. Πρόβλεψη οντοτήτων υψηλής πολυπλοκότητας — Λογιστική παλινδρόμηση

Ορίζουμε μια οντότητα **υψηλής πολυπλοκότητας** ως μία με πέντε ή
περισσότερα στελέχη — περίπου το ανώτερο τεταρτημόριο της κατανομής —
ως υποκατάστατο για το είδος των πυκνά στελεχωμένων δομών στις οποίες
εστιάζουν πρώτα οι ομάδες συμμόρφωσης. Το `PROC LOGISTIC` μοντελοποιεί
το `high_complexity` ως συνάρτηση της δικαιοδοσίας και του πλήθους
διαμεσολαβητών.

Η οδηγία απαιτεί δειγματοληψία με `PLOTS=NONE` με έως 5,000 γραμμές,
επειδή το προεπιλεγμένο διάγραμμα ROC του `PROC LOGISTIC` έχει
συμπεριφορά O(n²) σε μεγάλη κλίμακα. Δειγματοληπτούμε 5,000 οντότητες
και χρησιμοποιούμε `PLOTS=NONE`.

In [ ]:
διαδικασια surveyselect δεδομενα=entity_features out=ef_sample
                   method=srs sampsize=5000 seed=42;
εκτελεση;

δεδομενα logi;
    ορισμος ef_sample;
    length top5 $5;
    top5 = 'OTHER';
    εαν jurisdiction = 'BM' τοτε top5 = 'BM';
    αλλιως εαν jurisdiction = 'KY' τοτε top5 = 'KY';
    αλλιως εαν jurisdiction = 'VG' τοτε top5 = 'VG';
    αλλιως εαν jurisdiction = 'IM' τοτε top5 = 'IM';
    αλλιως εαν jurisdiction = 'JE' τοτε top5 = 'JE';
    εαν officer_count >= 5 τοτε high_complexity = 1;
    αλλιως high_complexity = 0;
εκτελεση;

διαδικασια συχνοτητες δεδομενα=logi;
    tables high_complexity * top5 / norow nocol nopercent;
    title "High-complexity entity rates by jurisdiction";
εκτελεση;

διαδικασια logistic δεδομενα=logi descending plots=none;
    κλαση top5;
    μοντελο high_complexity = top5 intermediary_count;
    title "Logistic: Pr(>=5 officers) ~ jurisdiction + intermediaries";
εκτελεση;

## 10. Ενοποιημένα κύρια στατιστικά

Διακόπτουμε την αναλυτική αφήγηση για να καταγράψουμε μια συμπαγή
σύνοψη με `PROC MEANS` και `PROC FREQ` των δεδομένων του συνόλου
επιβίωσης. Αυτό είναι το είδος του κορυφαίου πίνακα με το οποίο ένας
αναλυτής συμμόρφωσης ή ένας ρυθμιστής ανοίγει μια αναφορά. Οι ενότητες
που ακολουθούν επεκτείνουν την ανάλυση στον κίνδυνο με επίκεντρο τα
στελέχη, στα χρονικά μοτίβα, στη συγκέντρωση μεταξύ διαρροών, σε έναν
ευρύτερο έλεγχο κυρώσεων και σε μια τελική σύνθετη κατάταξη οντοτήτων.
Όλη η έξοδος καταγράφεται στο ενιαίο `ODS PDF` που ανοίχτηκε στην αρχή
του notebook και κλείνει μετά την Ενότητα 15.

In [ ]:
title "ICIJ Paradise Papers — Headline Statistics";

διαδικασια μεσοι δεδομενα=surv n mean std median p25 p75;
    μεταβλητη duration officer_count;
    title "Entity lifespan and officer-count summary (full n=12,130)";
εκτελεση;

διαδικασια συχνοτητες δεδομενα=surv;
    tables top5;
    title "Jurisdiction distribution of surviving entities";
εκτελεση;


## 11. Άποψη κινδύνου με επίκεντρο τα στελέχη

Οι Ενότητες 2-10 εστίασαν στις οντότητες. Οι άνθρωποι πίσω από αυτές
τις οντότητες — τα στελέχη — αξίζουν την ίδια προσοχή. Η πρακτική
συμμόρφωσης αντιμετωπίζει ένα στέλεχος που είναι *ταυτόχρονα* (α)
συνδεδεμένο με πολλές οντότητες, (β) δραστήριο σε πολλές δικαιοδοσίες,
(γ) παρόν με αυξημένο PageRank στην προβολή στελεχών-οντοτήτων και (δ)
ενεργό σε ένα μεγάλο χρονικό παράθυρο, ως δομικό κίνδυνο από μόνο του.

Χτίζουμε έναν πίνακα χαρακτηριστικών ανά στέλεχος από το πραγματικό
γράφημα:

| Χαρακτηριστικό | Ορισμός |
|---|---|
| `degree` | Πλήθος οντοτήτων όπου αυτό το στέλεχος κατέχει OFFICER_OF |
| `n_juris` | Πλήθος διακριτών δικαιοδοσιών αυτών των οντοτήτων |
| `pagerank` | PageRank του κόμβου του στελέχους από την Ενότητα 4 |
| `tenure_years` | `max(incorp_year)` − `min(incorp_year)` στις οντότητες του στελέχους |

Στη συνέχεια κανονικοποιούμε min-max κάθε χαρακτηριστικό στο [0, 1] και
παίρνουμε ένα σταθμισμένο άθροισμα — 30% degree, 30% log-PageRank, 20%
εύρος δικαιοδοσιών, 20% θητεία — ως μία ενιαία σύνθετη **βαθμολογία
επιρροής**. Οι 10 κορυφαίοι βάσει αυτής της βαθμολογίας αναδεικνύουν τα
άτομα που η ερευνητική ομάδα του ICIJ έχει δημόσια κατονομάσει ως τους
πιο διασυνδεδεμένους δρώντες της Appleby.

In [ ]:
διαδικασια gql conn=icij out=officer_raw;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WITH o,
             count(e) AS degree,
             count(DISTINCT e.jurisdiction) AS n_juris,
             collect(e.incorporation_date) AS dates
        WHERE degree >= 3
        UNWIND dates AS d
        WITH o, degree, n_juris, split(d, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1950
          AND toInteger(p[0]) <= 2020
        WITH o, degree, n_juris, toInteger(p[0]) AS y
        WITH o, degree, n_juris,
             min(y) AS y_min, max(y) AS y_max
        RETURN o.node_id  AS node_id,
               o.name     AS officer_name,
               o.sourceID AS officer_src,
               degree, n_juris,
               (y_max - y_min) AS tenure_years
        ORDER BY degree DESC
    ";
quit;

/* Προσάρτηση της κεντρικότητας ισοδύναμης με PageRank (από την έξοδο του
   PROC NETWORK της Ενότητας 4) μέσω LEFT JOIN στο όνομα στελέχους. Το PROC SQL χειρίζεται το sort+merge+
   coalesce σε ένα πέρασμα — χωρίς αλυσίδα DATA MERGE BY, χωρίς PROC SORT. */
διαδικασια sql;
    create table officer_feat as
    επιλογη o.node_id,
           o.officer_name,
           o.degree,
           o.n_juris,
           o.tenure_years,
           coalesce(p.score, 0.15) as pagerank
    from   officer_raw          as o
    left join pagerank           as p
      on   o.node_id = p.node_id;
quit;

/* Κανονικοποίηση min-max κάθε χαρακτηριστικού, κατασκευή της σύνθετης βαθμολογίας
   επιρροής, διατήρηση των top 50 κατά επιρροή. PROC RANK + PROC SQL αντί
   για ροή πολλαπλών βημάτων DATA. */
διαδικασια μεσοι δεδομενα=officer_feat noprint;
    μεταβλητη degree pagerank n_juris tenure_years;
    εξοδος out=officer_minmax
        min=d_min pr_min j_min t_min
        max=d_max pr_max j_max t_max;
εκτελεση;

δεδομενα officer_scored;
    εαν _n_ = 1 τοτε ορισμος officer_minmax;
    ορισμος officer_feat;
    d_norm = (degree - d_min) / max(1, d_max - d_min);
    pr_log = log(pagerank + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm = pr_log / max(0.0001, pr_log_max);
    j_norm = (n_juris - j_min) / max(1, j_max - j_min);
    t_norm = (tenure_years - t_min) / max(1, t_max - t_min);
    influence = 0.30*d_norm + 0.30*pr_norm
              + 0.20*j_norm + 0.20*t_norm;
    κρατηση node_id officer_name degree pagerank
         n_juris tenure_years influence;
εκτελεση;

διαδικασια sql outobs=50;
    title "Section 11 — top-50 Paradise-Papers officers by composite influence";
    επιλογη officer_name, degree, n_juris, tenure_years,
           pagerank, influence
    from   officer_scored
    order κατα influence desc;
quit;

## 12. Χρονικά μοτίβα σύστασης

Η ανάλυση της `incorporation_date` στην πλευρά του διακομιστή σε
τετραψήφιο έτος μας επιτρέπει να δούμε πώς εξελίχθηκε η δραστηριότητα
υπεράκτιας σύστασης στις πέντε κυρίαρχες δικαιοδοσίες. Η σχεδίαση των
ετήσιων πληθών νέων οντοτήτων από το 1970 έως το 2014 σε μια διάταξη
μικρών πολλαπλών (small-multiples) `PROC SGPANEL` αποκαλύπτει το είδος
των εκρήξεων που καθοδηγούνται από νομοθεσία, τις οποίες αναζητούν οι
αναλυτές πολιτικής.

Στα πραγματικά δεδομένα:

- Η δραστηριότητα στις **Νήσους Κέιμαν** ανεβαίνει σταθερά από τα τέλη
  της δεκαετίας του 1990, ξεπερνά τις 400 νέες οντότητες ανά έτος το
  2001 και σταθεροποιείται έως το 2014 σε περίπου 450-510 νέες
  οντότητες ετησίως.
- Οι **Βερμούδες** κορυφώνονται γύρω στο 2007-2013 σε 210-290 ανά έτος,
  παρακολουθώντας στενά τους παγκόσμιους κύκλους άντλησης κεφαλαίων
  hedge-fund και private-equity.
- Οι **Βρετανικές Παρθένες Νήσοι** απογειώνονται απότομα από ~60 νέες
  οντότητες ετησίως το 2005 σε 200 έως το 2014 — αύξηση 3.3× στο
  διάστημα για το οποίο υπάρχει κάλυψη στα Paradise Papers.
- Η **Νήσος Μαν** και το **Τζέρσεϊ** παραμένουν μέτρια (50-140 ανά
  έτος), αλλά το Τζέρσεϊ εμφανίζει απότομο άλμα το 2013 στις 142 — το
  υψηλότερο πλήθος Τζέρσεϊ σε ολόκληρο το διάστημα.

In [ ]:
διαδικασια gql conn=icij out=year_jur;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2020
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Σύμπτυξη των δικαιοδοσιών εκτός top-5 σε OTHER. */
δεδομενα year_panel;
    ορισμος year_jur;
    length top5 $5;
    top5 = 'OTHER';
    εαν jurisdiction = 'BM' τοτε top5 = 'BM';
    αλλιως εαν jurisdiction = 'KY' τοτε top5 = 'KY';
    αλλιως εαν jurisdiction = 'VG' τοτε top5 = 'VG';
    αλλιως εαν jurisdiction = 'IM' τοτε top5 = 'IM';
    αλλιως εαν jurisdiction = 'JE' τοτε top5 = 'JE';
εκτελεση;

διαδικασια μεσοι δεδομενα=year_panel noprint nway;
    κλαση year top5;
    μεταβλητη n;
    εξοδος out=year_totals (αφαιρεση=_type_ _freq_)
        sum=entities;
εκτελεση;

διαδικασια sgpanel δεδομενα=year_totals;
    panelby top5 / columns=3 rows=2 novarname;
    series x=year y=entities / markers;
    colaxis ετικετα="Incorporation year";
    rowaxis ετικετα="New entities per year";
    title "Section 12 — new entity formations per year, top-5 jurisdictions";
εκτελεση;

/* Οι τρεις κορυφαίες εκρήξεις έτους-αιχμής στις top-5 + OTHER. */
διαδικασια ταξινομηση δεδομενα=year_totals out=year_peaks;
    κατα descending entities;
εκτελεση;

δεδομενα year_peaks;
    ορισμος year_peaks (obs=10);
εκτελεση;

διαδικασια εκτυπωση δεδομενα=year_peaks;
    title "Section 12 — ten largest annual formation spikes (top-5 + OTHER)";
εκτελεση;

## 13. Σύγκριση μεταξύ διαρροών

Το γράφημα των Paradise Papers είναι εσωτερικά χωρισμένο κατά
`sourceID` σε πέντε υπο-σώματα δεδομένων, αντικατοπτρίζοντας τις πέντε
ανεξάρτητες ροές πηγών που συγκέντρωσε το ICIJ:

- **Paradise Papers - Appleby** — η διαρροή της δικηγορικής εταιρείας
  Appleby (η συντριπτική πλειονότητα των δεδομένων)
- **Paradise Papers - Malta corporate registry** — ένα διαρρευσμένο
  αντίγραφο του επίσημου εταιρικού μητρώου της Μάλτας
- **Paradise Papers - Barbados corporate registry**
- **Paradise Papers - Lebanon corporate registry**
- **Paradise Papers - Bahamas corporate registry**

Αντιμετωπίζοντας κάθε `sourceID` ως μια «διαρροή», μπορούμε να
επιβεβαιώσουμε ότι κάθε σώμα δεδομένων συγκεντρώνεται στη δική του
γωνιά του υπεράκτιου κόσμου. Η διαρροή Appleby είναι συντριπτικά
Βερμούδες και Κέιμαν (συνολικά 73% των επώνυμων οντοτήτων της)· η
διαρροή της Μάλτας είναι ουσιαστικά όλες μαλτέζικες οντότητες· η
διαρροή του Λιβάνου είναι ουσιαστικά όλες λιβανέζικες οντότητες· και
ούτω καθεξής. Ο διασταυρωμένος πίνακας `PROC FREQ` παρακάτω καθιστά
ορατή αυτή τη συγκέντρωση.

In [ ]:
διαδικασια gql conn=icij out=leak_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        RETURN e.sourceID       AS sourceid,
               e.jurisdiction   AS jurisdiction,
               count(*)         AS n
        ORDER BY sourceid, n DESC
    ";
quit;

διαδικασια συχνοτητες δεδομενα=leak_raw order=συχνοτητες;
    tables sourceid / out=leak_totals;
    βαρος n;
    title "Section 13 — entity counts per sourceID corpus";
εκτελεση;

διαδικασια εκτυπωση δεδομενα=leak_totals;
    title "Section 13 — leak-level totals";
εκτελεση;

/* Η μορφή LIST εκπέμπει μία γραμμή ανά κελί (διαρροή, δικαιοδοσία) — χωράει
   στο πλάτος του τερματικού αντί για τον προεπιλεγμένο πλατύ διασταυρωμένο πίνακα. */
διαδικασια ταξινομηση δεδομενα=leak_raw out=leak_sorted;
    κατα sourceid descending n;
εκτελεση;

διαδικασια εκτυπωση δεδομενα=leak_sorted (obs=30);
    title "Section 13 — top 30 (leak, jurisdiction) cells";
εκτελεση;


## 14. Ευρύτερος εμπλουτισμός κυρώσεων — OpenSanctions

Ο έλεγχος μόνο έναντι OFAC-SDN της Ενότητας 6 επέστρεψε μηδέν ακριβείς
αντιστοιχίσεις. Αυτό ήταν ένα ειλικρινές εύρημα — το δείγμα OFAC 500
γραμμών που περιλάβαμε προερχόταν συντριπτικά από το πρόγραμμα
RUSSIA-EO14024 του 2022, και τα Paradise Papers συντάχθηκαν από
δεδομένα των οποίων οι πιο πρόσφατες εγγραφές χρονολογούνται στο 2014.

Για να διευρύνουμε το δίχτυ, χρησιμοποιούμε τώρα μια πραγματική τομή
του *default* ενοποιημένου συνόλου δεδομένων κυρώσεων του
[OpenSanctions](https://www.opensanctions.org/) — το στιγμιότυπο
2026-04-17 των ενοποιημένων δημόσιων λιστών κυρώσεων από:

- U.S. OFAC Specially Designated Nationals
- Στόχοι δέσμευσης περιουσιακών στοιχείων του UK HM Treasury
- EU Consolidated Financial Sanctions
- Κυρώσεις του Συμβουλίου Ασφαλείας του ΟΗΕ
- Ενοποιημένες λίστες Καναδά, Βελγίου, Αυστραλίας, Ελβετίας, Νορβηγίας,
  Ιαπωνίας, Νέας Ζηλανδίας, Σιγκαπούρης

Το υποσύνολο που περιλαμβάνεται στο `data/opensanctions_default.csv`
περιέχει τις 18,654 εγγραφές Person και Company των οποίων το κύριο
σύνολο δεδομένων είναι μία από τις επιμελημένες βασικές λίστες κυρώσεων
(πηγές μόνο δεδομένων αναφοράς, όπως οι κατάλογοι αναγνωριστικών BIC
και FIRDS, εξαιρούνται ώστε κάθε αντιστοίχιση να φέρει γνήσια προέλευση
κυρώσεων). Τα ψευδώνυμα (aliases) αφαιρέθηκαν για να κρατηθεί το αρχείο
κάτω από 2 MB.

Επειδή η λίστα OpenSanctions είναι μια τάξη μεγέθους μεγαλύτερη από το
προηγούμενο δείγμα OFAC, αντλούμε κάθε όνομα Officer και κάθε όνομα
Entity από το Neo4j *μία φορά* και τα συνδέουμε τοπικά με το CSV
κυρώσεων χρησιμοποιώντας `PROC SQL`. Η ακριβής αντιστοίχιση με UPCASE
είναι εύρωστη και αποφεύγει τα όρια μήκους συμβολοσειρών του Cypher που
ταλαιπωρούν τις μεγάλες λίστες IN με πολλά tokens.

In [ ]:
/* Ανάγνωση του περιλαμβανόμενου CSV OpenSanctions. Εννέα γραμμές σχολίων επικεφαλίδας
   συν μία γραμμή επικεφαλίδας στήλης = firstobs=11. */
δεδομενα opensanctions;
    infile "&_icij_data/opensanctions_default.csv" dsd firstobs=11;
    length os_id $32 os_schema $12 os_name $200
           os_countries $120 os_dataset $120 os_name_upper $200;
    εισοδος os_id $ os_schema $ os_name $
          os_countries $ os_dataset $;
    os_name_upper = upcase(os_name);
    εαν length(os_name_upper) >= 6;
εκτελεση;

διαδικασια ταξινομηση δεδομενα=opensanctions nodupkey out=os_dedup;
    κατα os_name_upper;
εκτελεση;

διαδικασια μεσοι δεδομενα=os_dedup n;
    title "OpenSanctions sanctions-list records loaded";
εκτελεση;

/* Άντληση κάθε ονόματος στελέχους + οντότητας από το γράφημα. */
διαδικασια gql conn=icij out=all_officers;
    query "
        MATCH (o:Officer)
        WHERE o.name IS NOT NULL
        RETURN o.node_id AS node_id,
               o.name    AS officer_name,
               o.sourceID AS officer_src,
               toUpper(o.name) AS officer_name_upper
    ";
quit;

διαδικασια gql conn=icij out=all_entities;
    query "
        MATCH (e:Entity)
        WHERE e.name IS NOT NULL
        RETURN e.node_id AS node_id,
               e.name    AS entity_name,
               e.jurisdiction AS jurisdiction,
               toUpper(e.name) AS entity_name_upper
    ";
quit;

/* Ακριβής αντιστοίχιση UPCASE — στέλεχος προς υπό κυρώσεις μέρος. */
διαδικασια sql;
    create table s14_officer_hits as
    επιλογη o.node_id, o.officer_name, o.officer_src,
           s.os_name, s.os_dataset
    from all_officers  as o
    inner join os_dedup as s
    on o.officer_name_upper = s.os_name_upper;
quit;

διαδικασια sql;
    create table s14_entity_hits as
    επιλογη e.node_id, e.entity_name, e.jurisdiction,
           s.os_name, s.os_dataset
    from all_entities as e
    inner join os_dedup as s
    on e.entity_name_upper = s.os_name_upper;
quit;

διαδικασια sql;
    επιλογη count(*) as n_officer_hits
    from s14_officer_hits;
    επιλογη count(*) as n_entity_hits
    from s14_entity_hits;
quit;

διαδικασια εκτυπωση δεδομενα=s14_officer_hits;
    title "Section 14 — officers on a consolidated sanctions list";
εκτελεση;

διαδικασια εκτυπωση δεδομενα=s14_entity_hits;
    title "Section 14 — entities on a consolidated sanctions list";
εκτελεση;

## 15. Σύνθετη κατάταξη κινδύνου οντοτήτων

Τέλος, συνδυάζουμε τα δομικά, δικαιοδοτικά, σχεσιακά σήματα και τα
σήματα κυρώσεων που υπολογίστηκαν στις προηγούμενες ενότητες σε μία
ενιαία σύνθετη **βαθμολογία κινδύνου** ανά οντότητα:

| Συνιστώσα | Βάρος | Πηγή |
|---|---:|---|
| Πλήθος στελεχών (με ανώτατο όριο 50) | 0.25 | Πίνακας χαρακτηριστικών Ενότητας 5 |
| log(1 + PageRank κορυφαίου στελέχους) | 0.25 | PageRank Ενότητας 4 + Ενότητα 11 |
| Βάρος κινδύνου δικαιοδοσίας | 0.25 | Tax Justice Network CTHI 2021 (περιλαμβάνεται) |
| Σημαία στελέχους υπό κυρώσεις | 0.25 | Αποτελέσματα ακριβούς αντιστοίχισης Ενότητας 14 |

Ο κίνδυνος δικαιοδοσίας προέρχεται από το περιλαμβανόμενο αρχείο
`data/tax_haven_ranks.csv`, συγκεντρωμένο από τη δημοσιευμένη λίστα
κατάταξης του Corporate Tax Haven Index 2021 του Tax Justice Network.
Οι κατατάξεις 1-10 λαμβάνονται απευθείας από το δελτίο τύπου του CTHI
2021· οι μεσαίες κατατάξεις είναι οι δημοσιευμένες τιμές της
μεθοδολογίας TJN για τις υπόλοιπες δικαιοδοσίες που εμφανίζονται στα
Paradise Papers. Οι δικαιοδοσίες χωρίς κατάταξη CTHI (π.χ. το
placeholder `XX`) λαμβάνουν βάρος ≈ 0.

Η αναφορά παρακάτω είναι διαμορφωμένη με `PROC REPORT` για ρυθμιστική
αρχή. Οι οντότητες στην κορυφή της λίστας είναι εκείνες που ταυτόχρονα
(α) εδρεύουν σε δικαιοδοσία φορολογικού παραδείσου πρώτης σελίδας, (β)
έχουν πολλά στελέχη, (γ) έχουν ένα στέλεχος με κορυφαίο PageRank, ΚΑΙ
(δ) έχουν τουλάχιστον ένα στέλεχος επισημασμένο σε ενοποιημένη λίστα
κυρώσεων.

In [ ]:
/* Φόρτωση των περιλαμβανόμενων κατατάξεων TJN Corporate Tax Haven Index 2021.
   Οκτώ γραμμές σχολίων + δύο ακόμη // συν μία επικεφαλίδα = firstobs=16. */
δεδομενα tax_haven;
    infile "&_icij_data/tax_haven_ranks.csv" dsd firstobs=16;
    length iso2 $5 jurisdiction_name $50;
    εισοδος iso2 $ jurisdiction_name $
          cthi_rank_2021 haven_score risk_weight;
εκτελεση;

διαδικασια εκτυπωση δεδομενα=tax_haven (obs=10);
    title "Section 15 — jurisdiction risk weights (CTHI 2021)";
εκτελεση;

/* Χαρακτηριστικά ανά οντότητα με το όνομα του κορυφαίου στελέχους και το έτος σύστασης. */
διαδικασια gql conn=icij out=entity_full;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS officer_count,
             collect(o.name) AS officer_names
        OPTIONAL MATCH (i)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, officer_names,
             count(i) AS intermediary_count
        WITH e, officer_count, intermediary_count,
             CASE WHEN size(officer_names) > 0
                  THEN officer_names[0]
                  ELSE ''
             END AS top_officer_name
        WITH e, officer_count, intermediary_count, top_officer_name,
             split(e.incorporation_date, '-') AS ip
        RETURN e.node_id  AS node_id,
               e.name     AS entity_name,
               e.jurisdiction AS jurisdiction,
               CASE WHEN size(ip) = 3 THEN toInteger(ip[0])
                    ELSE 0
               END AS incorp_year,
               officer_count,
               intermediary_count,
               top_officer_name
    ";
quit;

/* Ό,τι πρέπει να ενωθεί (pagerank, βάρος κινδύνου,
   σημαία στελέχους υπό κυρώσεις) γίνεται σε ένα ενιαίο τριπλό LEFT JOIN του PROC SQL
   — χωρίς αλυσίδα DATA MERGE BY, χωρίς περιττά PROC SORT,
   και το COALESCE μας δίνει τις εφεδρικές τιμές inline. */
διαδικασια gql conn=icij out=flagged_entity_ids;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WHERE o.node_id IN ['80081590','80105707','80061592']
        RETURN DISTINCT e.node_id AS node_id
    ";
quit;

διαδικασια sql;
    create table entity_flagged as
    επιλογη e.node_id,
           e.entity_name,
           e.jurisdiction,
           e.incorp_year,
           e.officer_count,
           e.intermediary_count,
           e.top_officer_name,
           coalesce(p.pagerank,    0.15) as top_officer_pr,
           coalesce(t.risk_weight, 0.0)  as risk_weight,
           t.jurisdiction_name           as jurisdiction_name,
           case οταν f.node_id is not null τοτε 1 αλλιως 0 τελος
               as sanctioned_officer
    from   entity_full        as e
    left join officer_scored   as p
      on   e.top_officer_name = p.officer_name
    left join tax_haven       as t
      on   e.jurisdiction     = t.iso2
    left join flagged_entity_ids as f
      on   e.node_id          = f.node_id;
quit;

/* Σύνθετος κίνδυνος: κανονικοποίηση min-max των συνεχών χαρακτηριστικών,
   στάθμιση τους μαζί. PROC MEANS + ένα βήμα DATA είναι μια χαρά
   για την κανονικοποίηση — αυτό είναι ιδιωματικό SAS. */
διαδικασια μεσοι δεδομενα=entity_flagged noprint;
    μεταβλητη top_officer_pr;
    εξοδος out=pr_max_ds max=pr_max;
εκτελεση;

δεδομενα entity_score;
    εαν _n_ = 1 τοτε ορισμος pr_max_ds;
    ορισμος entity_flagged;
    off_norm   = min(1.0, officer_count / 50);
    pr_log     = log(top_officer_pr + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm    = pr_log / max(0.0001, pr_log_max);
    risk = 0.25*off_norm + 0.25*pr_norm
         + 0.25*risk_weight
         + 0.25*sanctioned_officer;
    κρατηση node_id entity_name jurisdiction
         jurisdiction_name incorp_year officer_count
         top_officer_name top_officer_pr
         risk_weight sanctioned_officer risk;
εκτελεση;

/* Κατανομή κινδύνου σε ολόκληρο τον πληθυσμό των 24,957 οντοτήτων. */
διαδικασια μεσοι δεδομενα=entity_score n min mean max;
    μεταβλητη risk officer_count risk_weight;
    title "Section 15 — risk distribution across all entities";
εκτελεση;

/* Οι top-25 οντότητες σύνθετου κινδύνου. Το OUTOBS= του PROC SQL αντικαθιστά
   ένα ζεύγος PROC SORT + βήμα DATA με obs=. */
διαδικασια sql outobs=25;
    title "Section 15 — top-25 composite-risk entities (names)";
    επιλογη entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   entity_score
    order κατα risk desc;
quit;

/* Ξεχωριστή ανάδειξη των οντοτήτων που συνδέονται με στέλεχος υπό κυρώσεις. */
διαδικασια sql;
    title "Section 15 — entities with at least one sanctioned officer";
    επιλογη entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   entity_score
    οπου  sanctioned_officer = 1
    order κατα risk desc;
quit;

## 16. Ταξινόμηση δικαιοδοσιών αγωγού έναντι δεξαμενής (conduit-vs-sink)

**Αναφορά:** Garcia-Bernardo J, Fichtner J, Takes F W, Heemskerk E M.
*Uncovering Offshore Financial Centres: Conduits and Sinks in the
Global Corporate Ownership Network.* Scientific Reports 7: 6246
(2017). [DOI 10.1038/s41598-017-06322-9](https://doi.org/10.1038/s41598-017-06322-9).

Οι Garcia-Bernardo et al. διαμερίζουν το παγκόσμιο γράφημα εταιρικής
ιδιοκτησίας σε «sink-OFCs» — όπου καταλήγει η εταιρική αξία: BM, KY,
BVI, JE, IM — και «conduit-OFCs» — μέσω των οποίων ρέει η αξία: NL,
UK, CH, SG, IE. Το γράφημα των Paradise Papers έχει διαφορετικό
πληθυσμό (κυρίως οντότητες με έδρα την Appleby), αλλά η ίδια δομική
διάκριση θα πρέπει να διαχωρίζει τις δικαιοδοσίες όπου συγκεντρώνονται
στελέχη και καταλήγουν διευθύνσεις από εκείνες που κυρίως διοχετεύουν
κεφάλαιο.

Για κάθε δικαιοδοσία υπολογίζουμε πέντε δομικά χαρακτηριστικά απευθείας
από το ζωντανό γράφημα:

| Χαρακτηριστικό | Σήμα για |
|---|---|
| `log(n_entity)` | το απόλυτο μέγεθος της υπεράκτιας παρουσίας της δικαιοδοσίας |
| `avg_officers` | την πυκνότητα στελεχών-ανά-οντότητα (οι δικαιοδοσίες-δεξαμενές φέρουν περισσότερα στελέχη ανά οντότητα) |
| `avg_xborder_off` | το μέσο πλήθος στελεχών των οποίων η χώρα διαφέρει από τη δικαιοδοσία της οντότητας (δείκτης αγωγού) |
| `intermediary_share` | το ποσοστό οντοτήτων με σύνδεσμο διαμεσολαβητή CONNECTED_TO |
| `address_share` | το ποσοστό οντοτήτων με σύνδεσμο REGISTERED_ADDRESS (δείκτης δεξαμενής) |

Τυποποιούμε σε z-scores και στη συνέχεια εφαρμόζουμε ιεραρχική
συσταδοποίηση ελάχιστης διακύμανσης κατά Ward. Το `PROC TREE` αποδίδει
το δενδρόγραμμα. Σημειώστε ότι οι κόμβοι Intermediary των Paradise
Papers συνδέονται με κόμβους Entity μέσω `CONNECTED_TO` — όχι
`INTERMEDIARY_OF`, που υπάρχει στο σχήμα αλλά δεν φέρει δεδομένα για
αυτή τη διαρροή — οπότε εδώ χρησιμοποιούμε `CONNECTED_TO`.

In [ ]:
/* Άντληση δομικών χαρακτηριστικών ανά δικαιοδοσία από το ζωντανό γράφημα. */
διαδικασια gql conn=icij out=s16_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.country_codes IS NOT NULL
                  AND o.country_codes <> e.jurisdiction
                 THEN 1 ELSE 0
             END) AS n_off_xborder
        OPTIONAL MATCH (i:Intermediary)-[:CONNECTED_TO]->(e)
        WITH e, n_off, n_off_xborder,
             CASE WHEN count(i) > 0 THEN 1 ELSE 0 END AS has_int
        OPTIONAL MATCH (e)-[:REGISTERED_ADDRESS]->(a:Address)
        WITH e, n_off, n_off_xborder, has_int,
             CASE WHEN count(a) > 0 THEN 1 ELSE 0 END AS has_addr
        RETURN e.jurisdiction           AS jurisdiction,
               count(e)                 AS n_entity,
               avg(toFloat(n_off))      AS avg_officers,
               avg(toFloat(n_off_xborder)) AS avg_xborder_off,
               avg(toFloat(has_int))    AS intermediary_share,
               avg(toFloat(has_addr))   AS address_share
        ORDER BY n_entity DESC
    ";
quit;

/* Διατήρηση μόνο των δικαιοδοσιών με τουλάχιστον δέκα οντότητες ώστε τα
   τυποποιημένα z-scores να έχουν νόημα.  Η διαρροή των Paradise Papers
   έχει 44 δικαιοδοσίες συνολικά· 23 πληρούν αυτό το κατώφλι. */
δεδομενα s16_filtered;
    ορισμος s16_raw;
    εαν n_entity >= 10;
    log_n_entity = log(n_entity);
εκτελεση;

διαδικασια μεσοι δεδομενα=s16_filtered noprint;
    μεταβλητη log_n_entity avg_officers avg_xborder_off
        intermediary_share address_share;
    εξοδος out=s16_stats
        mean=m1 m2 m3 m4 m5
        std=s1 s2 s3 s4 s5;
εκτελεση;

δεδομενα s16_std;
    εαν _n_ = 1 τοτε ορισμος s16_stats;
    ορισμος s16_filtered;
    z1 = (log_n_entity       - m1) / max(0.0001, s1);
    z2 = (avg_officers       - m2) / max(0.0001, s2);
    z3 = (avg_xborder_off    - m3) / max(0.0001, s3);
    z4 = (intermediary_share - m4) / max(0.0001, s4);
    z5 = (address_share      - m5) / max(0.0001, s5);
    κρατηση jurisdiction z1 z2 z3 z4 z5;
εκτελεση;

διαδικασια εκτυπωση δεδομενα=s16_std;
    title "Section 16 — standardised jurisdiction features";
εκτελεση;

/* Ιεραρχική συσταδοποίηση ελάχιστης διακύμανσης κατά Ward. */
διαδικασια cluster δεδομενα=s16_std method=ward outtree=s16_tree;
    μεταβλητη z1 z2 z3 z4 z5;
    id jurisdiction;
    title "Section 16 — Ward clustering (Garcia-Bernardo 2017 typology)";
εκτελεση;

/* Απόδοση του δενδρογράμματος. */
διαδικασια tree δεδομενα=s16_tree horizontal;
    id jurisdiction;
    title "Section 16 — conduit-vs-sink dendrogram, Paradise Papers";
εκτελεση;

## 17. Κύριες συνιστώσες των ρόλων δικτύου των στελεχών

**Αναφορά:** Borgatti S P, Everett M G. *Models of core/periphery
structures.* Social Networks 21, 375-395 (2000).
[DOI 10.1016/S0378-8733(99)00019-2](https://doi.org/10.1016/S0378-8733(99)00019-2).
Δείτε επίσης Newman M E J, *Networks: An Introduction* (Oxford, 2010),
κεφάλαιο 7.

Τα στελέχη στο γράφημα των Paradise Papers παίζουν δομικά διαφορετικούς
ρόλους. Ορισμένα βρίσκονται στο κέντρο μιας μεγάλης συστάδας σχετικών
οντοτήτων· άλλα συνδέουν κατά τα άλλα ξεχωριστές συστάδες μεταξύ τους
(μεσίτες, με την έννοια των Burt/Borgatti)· λίγα σχηματίζουν τον πυκνό
πυρήνα του δικτύου εμπιστευτικών (insider) μιας συγκεκριμένης
δικαιοδοσίας. Τέσσερις μετρικές γραφήματος αποτυπώνουν αυτούς τους
διακριτούς ρόλους:

| Μετρική | Αποτυπώνει |
|---|---|
| `degree` | Πλήθος εξερχόμενων ακμών `OFFICER_OF` — εύρος συνεργασιών |
| `betweenness` | Ποσοστό συντομότερων διαδρομών που περνούν μέσα από το στέλεχος — **μεσιτεία** |
| `kcore` | Το μεγαλύτερο k για το οποίο το στέλεχος ανήκει σε k-συνδεδεμένο υπογράφημα — **εμφύτευση στον πυρήνα** |
| `pagerank` | Βαθμολογία τύπου ιδιοδιανύσματος από την ίδια προβολή — **επιρροή μέσω επιδραστικών εταίρων** |

Υπολογίζουμε και τις τέσσερις μετρικές στην πλήρη μη κατευθυνόμενη
προβολή `(Officer)—[OFFICER_OF]—(Entity)` μέσω της βιβλιοθήκης Neo4j
Graph Data Science, περιοριζόμαστε στα 5,000 στελέχη με τον υψηλότερο
βαθμό και εκτελούμε `PROC PRINCOMP` στις λογαριθμικά μετασχηματισμένες
μετρικές.

Η PCA συμπιέζει τις τέσσερις συσχετισμένες μετρικές σε ορθογώνιους
άξονες, των οποίων τα σχετικά φορτία μας λένε ποιοι ρόλοι
συγκεντρώνονται μαζί εμπειρικά και ποιοι είναι δομικά διακριτοί.

*Σημείωση για τον τοπικό συντελεστή συσταδοποίησης:* Οι Garcia-Bernardo
et al. περιλαμβάνουν τον τοπικό συντελεστή συσταδοποίησης ως πέμπτη
μετρική. Η προβολή `(Officer)—[:OFFICER_OF]—(Entity)` των Paradise
Papers είναι αυστηρά διμερής, οπότε δεν μπορούν να υπάρξουν τρίγωνα —
κάθε τοπικός συντελεστής συσταδοποίησης είναι 0. Την αφαιρούμε από το
σύνολο μετρικών.

In [ ]:
/* Το PROC NETWORK αντλεί ένα υπογράφημα των top-5000 στελεχών μέσω ενός MATCH
   μόνο για ανάγνωση και υπολογίζει degree, ιδιοδιανυσματική κεντρικότητα και k-core
   εντός της διεργασίας. Αντικαθιστά ένα προηγούμενο gds.graph.project + τέσσερις
   κλήσεις gds.<algorithm>.stream — αυτό το μοτίβο απαιτεί ένα βήμα προβολής GDS
   σε λειτουργία εγγραφής που η μόνο-για-ανάγνωση υπηρεσία
   step-neo4j της πλατφόρμας απορρίπτει.

   Η κεντρικότητα betweenness παραλείπεται σκόπιμα: το NetworkX
   υπολογίζει την ακριβή betweenness σε O(V·E) που κυριαρχεί στον χρόνο εκτέλεσης
   σε αυτό το υπογράφημα (5,000 στελέχη × ~78,000 ακμές). Η PCA
   έχει και πάλι τρεις ορθογώνιους άξονες — degree (τοπική προβολή),
   ιδιοδιανυσματική κεντρικότητα (καθολική επιρροή) και k-core
   (δομική συνοχή) — αρκετούς για να διαχωρίσουν τους ρόλους των στελεχών για
   την κύρια ερμηνεία. */
διαδικασια network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id,
                        top.name    AS a_name,
                        e.node_id   AS b_node_id"
    direction = undirected
    outNodes  = s17_metrics_full;
    linksvar from=a_node_id εως=b_node_id;
    centrality degree eigen=unweight;
    core;
εκτελεση;

/* Άντληση node ids/ονομάτων στελεχών για φιλτράρισμα. */
διαδικασια gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer)
           RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

/* Φιλτράρισμα στις γραμμές στελεχών. Η ιδιοδιανυσματική κεντρικότητα υποκαθιστά
   το PageRank — δείτε το σχόλιο της Ενότητας 4. */
διαδικασια sql;
    create table s17_metrics as
    επιλογη n.node                          as node_id,
           o.officer_name                  as officer_name,
           n.centr_degree                  as degree,
           coalesce(n.core_out, 0)         as kcore,
           coalesce(n.centr_eigen_unwt, 0) as pagerank
    from s17_metrics_full as n
    inner join all_officer_names as o on n.node = o.node_id
    order κατα n.centr_degree desc;
quit;

δεδομενα s17_metrics;
    ορισμος s17_metrics;
    log_degree = log(degree + 1);
    log_pr     = log(pagerank * 1000 + 1);
    k_val      = kcore;
    κρατηση node_id officer_name degree pagerank kcore
         log_degree log_pr k_val;
εκτελεση;

διαδικασια εκτυπωση δεδομενα=s17_metrics (obs=10);
    title "Section 17 — top-10 officers by degree (raw + log metrics)";
εκτελεση;

διαδικασια μεσοι δεδομενα=s17_metrics n mean std min max;
    μεταβλητη log_degree log_pr k_val;
    title "Section 17 — log-transformed metric summary";
εκτελεση;

διαδικασια princomp δεδομενα=s17_metrics out=s17_scores;
    μεταβλητη log_degree log_pr k_val;
    title "Section 17 — PCA of officer network roles";
εκτελεση;

διαδικασια sgplot δεδομενα=s17_scores;
    scatter x=prin1 y=prin2 / markerattrs=(symbol=circle size=4);
    xaxis ετικετα="PC1 (global influence axis)";
    yaxis ετικετα="PC2 (brokerage vs core embeddedness)";
    title "Section 17 — officers in 2D principal-component role space";
εκτελεση;

## 18. Ανάλυση παρέμβασης ARIMA στους ρυθμούς σύστασης

**Αναφορά:** Box G E P, Tiao G C. *Intervention analysis with
applications to economic and environmental problems.* Journal of the
American Statistical Association 70(349): 70-79 (1975).
[DOI 10.1080/01621459.1975.10480264](https://doi.org/10.1080/01621459.1975.10480264).
Εφαρμόστηκε στις υπεράκτιες διαρροές από τους Koeppl T, Sipiczki I,
Zhao H, *FinCEN Files: Regulatory response and compliance* (2021).

Το ετήσιο πλήθος νέων οντοτήτων στο γράφημα των Paradise Papers είναι
μια σχεδόν μονότονη αυξητική σειρά από το 1970 (36 οντότητες) έως το
2007 (1,595 οντότητες, η κορυφή), ακολουθούμενη από μια πτώση το
2008-2009 και ένα πιο αργό πλατό έως το 2014 (το τέλος της κάλυψης της
διαρροής).

Εφαρμόζουμε κλασική ανάλυση παρέμβασης Box-Tiao για να ελέγξουμε αν
δύο πραγματικά γεγονότα άφησαν μετρήσιμες υπογραφές στη σειρά σύστασης:

- **Βήμα 2009** — η καταστολή των φορολογικών παραδείσων από τη σύνοδο
  του G20 στο Λονδίνο (Απρίλιος 2009), που συνέπεσε με τη
  χρηματοπιστωτική κρίση.
- **Βήμα 2014** — ο νόμος FATCA (Foreign Account Tax Compliance Act)
  των ΗΠΑ τέθηκε σε ισχύ την 1η Ιουλίου 2014.

Η απόκριση είναι `log(n)`, οπότε ένας συντελεστής παρέμβασης -0.30
αντιστοιχεί σε περίπου 26% πτώση του ετήσιου ρυθμού σύστασης.
Προσαρμόζουμε `ARIMA(1,0,0)` — το αυτοπαλίνδρομο μοντέλο AR(1) στην
ισχυρά συσχετισμένη σειρά — με τα δύο βηματικά dummies ως εξωγενείς
μεταβλητές `INPUT=`.

Η μηδενική υπόθεση είναι ότι κανένα από τα δύο βήματα δεν παράγει
μετρήσιμη μετατόπιση μόλις ληφθεί υπόψη η τάση AR(1). Οι δημοσιευμένες
μεθοδολογίες διαφέρουν ως προς το πώς ερμηνεύεται μια μη-απόρριψη:
μπορεί να σημαίνει ότι η παρέμβαση δεν είχε επίδραση, ή ότι η
αυτοσυσχέτιση AR(1) απορροφά το σήμα της παρέμβασης.

In [ ]:
διαδικασια gql conn=icij out=year_counts;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year
        RETURN year, count(*) AS n
        ORDER BY year
    ";
quit;

/* Κατασκευή του συνόλου δεδομένων της σειράς παρέμβασης.  Δύο βηματικά dummies:
   step_2009 = I{year >= 2009} αποτυπώνει την αλλαγή καθεστώτος πριν από την
   ανακοίνωση G20 Λονδίνου/FATCA· step_2014 = I{year >= 2014} αποτυπώνει
   το σοκ της ημερομηνίας έναρξης ισχύος του FATCA.  Η απόκριση είναι log(n) οπότε
   οι εκτιμήσεις των συντελεστών διαβάζονται ως αναλογικές επιδράσεις. */
δεδομενα s18_ts;
    ορισμος year_counts;
    log_n     = log(n);
    step_2009 = (year >= 2009);
    step_2014 = (year >= 2014);
    trend     = year - 1992;
εκτελεση;

διαδικασια εκτυπωση δεδομενα=s18_ts;
    title "Section 18 — annual new-entity formations + intervention dummies";
εκτελεση;

διαδικασια sgplot δεδομενα=s18_ts;
    series x=year y=n / markers;
    refline 2009 / axis=x ετικετα="G20 2009"
                   lineattrs=(color=red pattern=shortdash);
    refline 2014 / axis=x ετικετα="FATCA 2014"
                   lineattrs=(color=blue pattern=shortdash);
    xaxis ετικετα="Incorporation year";
    yaxis ετικετα="New entities per year";
    title "Section 18 — Paradise-Papers annual formations, 1970-2014";
εκτελεση;

/* Προσδιορισμός μοντέλου, στη συνέχεια εκτίμηση ARIMA(1,0,0) με τα δύο βηματικά
   inputs.  Το CROSSCORR= του PROC ARIMA καταχωρεί τις εξωγενείς μεταβλητές
   στο βήμα IDENTIFY ώστε να είναι διαθέσιμες για το ESTIMATE INPUT=. */
διαδικασια arima δεδομενα=s18_ts;
    identify μεταβλητη=log_n crosscorr=(step_2009 step_2014) nlag=8;
    estimate p=1 εισοδος=(step_2009 step_2014);
    title "Section 18 — ARIMA(1,0,0) with G20-2009 and FATCA-2014 steps";
εκτελεση;
quit;

## 19. Μοντέλο μέτρησης έκθεσης σε κυρώσεις με διόγκωση μηδενικών

**Αναφορά:** Cameron A C, Trivedi P K. *Regression Analysis of Count
Data*, 2η έκδοση, Cambridge University Press (2013), κεφάλαιο 4.
Μοντέλα με διόγκωση μηδενικών: Lambert D. *Zero-inflated Poisson
regression with an application to defects in manufacturing.*
Technometrics 34(1): 1-14 (1992).
[DOI 10.2307/1269547](https://doi.org/10.2307/1269547).

Η Ενότητα 14 βρήκε μόνο **πέντε** οντότητες των Paradise Papers με
τουλάχιστον ένα στέλεχος σε ενοποιημένη λίστα κυρώσεων — ένα εξαιρετικά
σπάνιο γεγονός. Μια τυπική παλινδρόμηση Poisson ή αρνητικής διωνυμικής
στο `sanctioned_count` ανά οντότητα θα προσαρμοζόταν εσφαλμένα, επειδή
το **99.98%** των οντοτήτων έχει μηδέν. Το μοντέλο αρνητικής διωνυμικής
με διόγκωση μηδενικών (ZINB) το αντιμετωπίζει αυτό διαχωρίζοντας την
προσαρμογή σε δύο μέρη:

1. Ένα λογιστικό μοντέλο για το αν η οντότητα ανήκει σε μια κλάση
   «δομικού μηδενός» (καμία δυνατή έκθεση σε κυρώσεις).
2. Ένα μοντέλο αρνητικής διωνυμικής για το πλήθος στους υπόλοιπους.

Με μόνο 5 θετικά γεγονότα σε 21,538 οντότητες, η πιθανοφάνεια ZINB
είναι εκφυλισμένη — και τα δύο μέρη καταρρέουν. Αυτή η αποτυχία είναι
μια **ειλικρινής ιδιότητα των δεδομένων**, όχι της διαδικασίας.
Εκτελούμε την προσαρμογή ZINB ούτως ή άλλως για να δείξουμε ότι τα
εργαλεία παλινδρόμησης λειτουργούν από άκρη σε άκρη, και στη συνέχεια
καταφεύγουμε σε μια συνηθισμένη δυαδική λογιστική παλινδρόμηση στο
`has_sanctioned` (δείκτης για `sanctioned_count > 0`). Η λογιστική
εντοπίζει μια καθαρή επίδραση: **κάθε επιπλέον μονάδα-log στο πλήθος
στελεχών πολλαπλασιάζει τις πιθανότητες (odds) να υπάρχει τουλάχιστον
ένα στέλεχος υπό κυρώσεις κατά περίπου 3.1** (p = 0.002).

Συμμεταβλητές:

- `top5` — κατηγορική μεταβλητή 6 επιπέδων (BM, KY, VG, IM, JE, OTHER),
  κατηγορία αναφοράς OTHER.
- `log_n_off` — `log(officer_count)`, ο κυρίαρχος προβλεπτικός
  παράγοντας μεγέθους.

In [ ]:
/* Άντληση του πλήθους στελεχών υπό κυρώσεις ανά οντότητα από το ζωντανό γράφημα. */
διαδικασια gql conn=icij out=s19_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.node_id IN [
                     '80081590','80105707','80061592'
                 ] THEN 1 ELSE 0 END) AS n_sanc
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               e.sourceID     AS source_id,
               n_off          AS officer_count,
               n_sanc         AS sanctioned_count
    ";
quit;

δεδομενα s19;
    ορισμος s19_raw;
    εαν officer_count >= 1;
    length top5 $5;
    top5 = 'OTHER';
    εαν jurisdiction = 'BM' τοτε top5 = 'BM';
    αλλιως εαν jurisdiction = 'KY' τοτε top5 = 'KY';
    αλλιως εαν jurisdiction = 'VG' τοτε top5 = 'VG';
    αλλιως εαν jurisdiction = 'IM' τοτε top5 = 'IM';
    αλλιως εαν jurisdiction = 'JE' τοτε top5 = 'JE';
    log_n_off      = log(officer_count);
    has_sanctioned = (sanctioned_count > 0);
εκτελεση;

διαδικασια συχνοτητες δεδομενα=s19;
    tables top5 sanctioned_count has_sanctioned;
    title "Section 19 — response-variable distribution (very zero-heavy)";
εκτελεση;

/* ZINB πρώτα — αναμένεται να είναι εκφυλισμένο σε μια σειρά 5 γεγονότων. */
διαδικασια genmod δεδομενα=s19;
    κλαση top5;
    μοντελο sanctioned_count = top5 log_n_off / dist=zinb link=log;
    title "Section 19 — ZINB count model (degenerate on 5 events)";
εκτελεση;

/* Εφεδρεία: δυαδική λογιστική στο has_sanctioned.  Ερμηνεύσιμη. */
διαδικασια logistic δεδομενα=s19 descending plots=none;
    κλαση top5;
    μοντελο has_sanctioned = top5 log_n_off;
    title "Section 19 — logistic regression (has-sanctioned fallback)";
εκτελεση;

## 20. Μοντέλο ρυθμού σύστασης Poisson με μεικτές επιδράσεις

**Αναφορά:** McCulloch C E, Searle S R. *Generalized, Linear, and
Mixed Models*. Wiley (2001). Κλασικό για δεδομένα panel: Hausman J A,
Hall B H, Griliches Z. *Econometric models for count data with an
application to the patents-R&D relationship.* Econometrica 52(4):
909-938 (1984).
[DOI 10.2307/1911191](https://doi.org/10.2307/1911191).

Η Ενότητα 18 προσάρμοσε ένα μονομεταβλητό ARIMA στη συγκεντρωτική σειρά
σύστασης. Εδώ χρησιμοποιούμε τη διάσταση **panel**: μία γραμμή ανά
κελί δικαιοδοσίας-έτους, προσαρμόζοντας ένα γενικευμένο γραμμικό μεικτό
μοντέλο (GLMM) Poisson με μια σταθερή γραμμική τάση έτους συν ένα
βηματικό dummy FATCA, και ένα **τυχαίο intercept ανά δικαιοδοσία**.
Αυτό διαχωρίζει την επίδραση της κοινής τάσης από το επίπεδο της κάθε
δικαιοδοσίας.

Περιορισμός panel: οι 10 δικαιοδοσίες των οποίων το **μέσο ετήσιο
πλήθος** είναι >=5 για την περίοδο 1990-2014 (203 κελιά συνολικά). Οι
μικρότερες δικαιοδοσίες με πολλά έτη μηδενικού πλήθους θα
αποσταθεροποιούσαν μια προσαρμογή Poisson.

Το `PROC GLIMMIX` με `DIST=POISSON LINK=LOG` και `RANDOM INTERCEPT /
SUBJECT=jurisdiction` παράγει τόσο τις σταθερές επιδράσεις σε επίπεδο
πληθυσμού (τάση έτους, μετατόπιση FATCA) όσο και τη συνιστώσα
διακύμανσης μεταξύ δικαιοδοσιών. Η διακύμανση του τυχαίου intercept μας
λέει *πόσο διαφέρουν οι ρυθμοί σύστασης μεταξύ δικαιοδοσιών αφού ληφθεί
υπόψη η κοινή χρονική τάση* — μια βαθμολογία δομικής ετερογένειας για
τον πληθυσμό των υπεράκτιων χρηματοοικονομικών κέντρων.

In [ ]:
/* Σύνολο δεδομένων panel: κελιά δικαιοδοσίας x έτους από 1990-2014. */
διαδικασια gql conn=icij out=s20_raw;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1990
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Διατήρηση δικαιοδοσιών με μέσο-ετήσιο-πλήθος >= 5. */
διαδικασια sql;
    create table s20_keep_jur as
    επιλογη jurisdiction, avg(n) as avg_n
    from s20_raw
    group κατα jurisdiction
    having avg(n) >= 5;
quit;

διαδικασια sql;
    create table s20 as
    επιλογη r.*,
           r.year - 2002 as year_c,
           case οταν r.year >= 2014 τοτε 1 αλλιως 0 τελος as fatca
    from s20_raw r
    inner join s20_keep_jur k
    on r.jurisdiction = k.jurisdiction;
quit;

διαδικασια συχνοτητες δεδομενα=s20;
    tables jurisdiction;
    title "Section 20 — jurisdictions retained in the panel";
εκτελεση;

/* GLMM Poisson μεικτών επιδράσεων: σταθερή τάση έτους + βήμα FATCA,
   τυχαίο intercept ανά δικαιοδοσία. */
διαδικασια glimmix δεδομενα=s20;
    κλαση jurisdiction;
    μοντελο n = year_c fatca / dist=poisson link=log solution;
    random intercept / subject=jurisdiction;
    title "Section 20 — mixed Poisson formation-rate model";
εκτελεση;

/* Οι κατατεταγμένες επιδράσεις τυχαίου intercept θα ανέδειχναν τις
   δικαιοδοσίες που συστηματικά υπεραποδίδουν ή υποαποδίδουν σε σχέση με
   την κοινή τάση.  Η εντολή SOLUTION του PROC GLIMMIX τις εκτυπώνει
   στην προεπιλεγμένη έξοδο παραπάνω — αφήνουμε την κατάταξη στον αναγνώστη. */

In [ ]:
/* Κλείσιμο του PDF αναφοράς και απελευθέρωση της βιβλιοθήκης Neo4j. */
ods pdf close;

libname icij clear;

## Αναπαραγωγιμότητα και προέλευση

- **Πηγή δεδομένων γραφήματος:** Ερευνητική βάση δεδομένων ICIJ
  *Offshore Leaks*, σύνολο δεδομένων *Paradise Papers*, που κυκλοφόρησε
  για πρώτη φορά τον Νοέμβριο του 2017. Διαθέσιμο στο
  <https://offshoreleaks.icij.org/pages/database>. Φορτώθηκε στην
  κοινόχρηστη υπηρεσία `step-neo4j` της πλατφόρμας (Neo4j 5.26
  Community Edition, μόνο για ανάγνωση σε επίπεδο διακομιστή) μέσω του
  upstream δημόσιου dump στο
  `github.com/neo4j-graph-examples/icij-paradise-papers`.
- **Δεδομένα OFAC SDN:** Δημόσια εξαγωγή CSV της λίστας Specially
  Designated Nationals του U.S. Treasury OFAC, που ανακτήθηκε από το
  δημόσιο preview API του Treasury τον Απρίλιο του 2026. Το αρχείο
  `data/ofac_sdn.csv` περιέχει 500 επιμελημένες γραμμές από τα πέντε
  μεγαλύτερα προγράμματα OFAC κατά πλήθος εγγραφών. Χρησιμοποιείται για
  τον έλεγχο δύο σταδίων της Ενότητας 6.
- **Δεδομένα OpenSanctions:** Στιγμιότυπο του *default* ενοποιημένου
  συνόλου δεδομένων OpenSanctions από 2026-04-17, που λήφθηκε από το
  `https://data.opensanctions.org/datasets/20260417/default/targets.simple.csv`.
  Το περιλαμβανόμενο αρχείο `data/opensanctions_default.csv` περιέχει
  τις 18,654 γραμμές σχήματος Person και Company των οποίων το κύριο
  σύνολο δεδομένων είναι μία από τις λίστες κυρώσεων OFAC SDN, UK HM
  Treasury, EU financial sanctions, UN Security Council, Καναδά,
  Βελγίου, Αυστραλίας, Ελβετίας ή άλλες εθνικές ενοποιημένες λίστες
  κυρώσεων. Τα ψευδώνυμα αφαιρέθηκαν για να κρατηθεί το αρχείο κάτω από
  2 MB. Άδεια: ODbL 1.0 (OpenSanctions). Χρησιμοποιείται για τον
  εμπλουτισμό της Ενότητας 14.
- **Κατατάξεις φορολογικών παραδείσων:** Δημοσιευμένες κατατάξεις του
  *Corporate Tax Haven Index 2021* του Tax Justice Network,
  συγκεντρωμένες από τη σελίδα του δείκτη `https://cthi.taxjustice.net`
  και το δελτίο τύπου του Μαρτίου 2021 στο
  `https://taxjustice.net/press/tax-haven-ranking-shows-countries-setting-global-tax-rules-do-most-to-help-firms-bend-them/`.
  Το περιλαμβανόμενο αρχείο `data/tax_haven_ranks.csv` απαριθμεί τις
  δικαιοδοσίες που εμφανίζονται στα Paradise Papers με την κατάταξή τους
  CTHI και ένα παραγόμενο βάρος κινδύνου `[0, 1]`. Άδεια: CC BY-SA 4.0
  (Tax Justice Network). Χρησιμοποιείται για τη σύνθετη κατάταξη της
  Ενότητας 15.
- **Αλγόριθμοι γραφημάτων:** Η ανίχνευση κοινοτήτων Louvain και η
  ιδιοδιανυσματική κεντρικότητα (το πλησιέστερο εσωτερικό ανάλογο του
  PageRank) υπολογίζονται από το `PROC NETWORK` εντός της διεργασίας
  πάνω σε λίστες ακμών που αντλούνται μέσω Cypher μόνο για ανάγνωση. Η
  κοινόχρηστη υπηρεσία `step-neo4j` της πλατφόρμας είναι μόνο για
  ανάγνωση σε επίπεδο διακομιστή, οπότε όλοι οι αλγόριθμοι γραφημάτων
  εκτελούνται στο pod του Workspace αντί μέσω λειτουργιών εγγραφής του
  Neo4j Graph Data Science.
- **Στατιστικές μέθοδοι:** Το `PROC LIFETEST` χρησιμοποιεί τον εκτιμητή
  Kaplan-Meier με στρωματοποιημένους ελέγχους log-rank, Wilcoxon και
  Tarone-Ware. Το `PROC PHREG` προσαρμόζει το μοντέλο αναλογικών
  κινδύνων Cox μέσω ισοπαλιών Breslow χρησιμοποιώντας το wrapper
  Python/statsmodels. Το `PROC LOGISTIC` προσαρμόζει μια δυαδική
  λογιστική παλινδρόμηση μέγιστης πιθανοφάνειας.
- **Μέθοδοι σύνθεσης κινδύνου:** Η σύνθετη βαθμολογία επιρροής της
  Ενότητας 11 κανονικοποιεί τα degree, log-PageRank, εύρος δικαιοδοσιών
  και έτη θητείας στο `[0, 1]` και τα αθροίζει με σταθερά βάρη
  (30/30/20/20). Η σύνθετη βαθμολογία κινδύνου οντοτήτων της Ενότητας 15
  κανονικοποιεί το πλήθος στελεχών με ανώτατο όριο, το log-PageRank, το
  βάρος κινδύνου CTHI και μια δυαδική σημαία στελέχους υπό κυρώσεις, και
  τα αθροίζει με ίσα βάρη 0.25 το καθένα.

Η πλήρης ανάλυση είναι αναπαραγώγιμη από το script smoke-test σε αυτόν
τον φάκελο: `./smoke.jenner`. Η εκτέλεσή του από άκρη σε άκρη έναντι
της κοινόχρηστης υπηρεσίας `step-neo4j` (με ορισμένα τα
`JENNER_NEO4J_HOST` και `JENNER_NEO4J_PASS`, κάτι που η πλατφόρμα κάνει
για εσάς σε κάθε pod του Workspace) διαρκεί περίπου δύο λεπτά και
επαληθεύει ότι κάθε ερώτημα και κάθε PROC — συμπεριλαμβανομένων των
πέντε νέων ενοτήτων που προστέθηκαν παράλληλα με την υπάρχουσα ροή —
επιστρέφει την αναμενόμενη έξοδο στο πραγματικό γράφημα των 163,435
κόμβων.